# LLM 빈칸 실습 모음 · Chapter 2~6

**문제 → 빈칸 코드 → 검증 코드 → 접힌 해설·정답** 순서입니다. 준비 셀을 한 번 실행한 뒤 원하는 문제부터 풀 수 있습니다. CPU PyTorch만 필요하며 다운로드·실제 LLM·GPT-2 토크나이저 설치는 없습니다.

원본 `???`만 `MISSING`으로 바꿔 문법적으로 실행 가능한 빈칸을 만들었습니다. **MISSING을 알맞은 코드로 교체**하세요. 따옴표 안 MISSING은 문자열 키, `.MISSING`은 메서드/필드, 단독 MISSING은 식/인자입니다. 원본 함수·클래스를 작은 solve 함수 안에 넣었으며 나머지 문맥은 제공합니다.

SyntaxError가 난 경우 이전 함수가 남을 수 있습니다. 수정 후 문제 셀부터 다시 실행하세요. PASS는 아래 더미 입력의 검사 결과이며 실제 GPT 품질이나 학습자 숙련을 보장하지 않습니다.

Chapter2 Tokenizer 비교 이후·다운로드는 제외했습니다. 해설은 먼저 시도한 뒤 펼치세요. 준비 셀의 보조 구현은 문제 간 의존성을 없애기 위한 제공 코드입니다. 출제 공식 답안 파일은 지정되지 않아 원본 힌트와 수식으로 참고 답안을 구성했습니다.

In [ ]:
import math
import copy
import torch
import torch.nn as nn  # torch.nn의 별명
from torch.utils.data import Dataset, DataLoader, TensorDataset
from types import SimpleNamespace
torch.set_num_threads(1)
MISSING = object()  # ???? 위치의 MISSING을 정답으로 교체하세요.

class DummyTokenizer:
    # UTF-8 byte를 번호로 쓰는 오프라인 더미입니다. 실제 GPT-2 BPE 결과가 아닙니다.
    def encode(self, text, allowed_special=None):
        return list(text.encode('utf-8'))
    def decode(self, ids):
        return bytes(ids).decode('utf-8')

class TinyLM(nn.Module):
    def __init__(self, vocab=11, dim=4):
        super().__init__()
        self.emb=nn.Embedding(vocab,dim)
        self.head=nn.Linear(dim,vocab)
    def forward(self,x): return self.head(self.emb(x))

class NextTokenModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.anchor=nn.Parameter(torch.zeros(()))
        self.calls=[]
    def forward(self,idx):
        self.calls.append((idx.detach().clone(),torch.is_grad_enabled()))
        return torch.nn.functional.one_hot((idx+1)%7,7).float()*4 + self.anchor

class TinyClassifier(nn.Module):
    def __init__(self, dim=4):
        super().__init__()
        self.emb=nn.Embedding(11,dim)
        self.trf_blocks=nn.Sequential(nn.Linear(dim,dim),nn.Linear(dim,dim))
        self.final_norm=nn.LayerNorm(dim)
        self.out_head=nn.Linear(dim,11)
    def forward(self,x):
        return self.out_head(self.final_norm(self.trf_blocks(self.emb(x))))

class MiniFrame:
    # pandas의 iloc[index]['Label'] 인터페이스만 제공하는 더미입니다.
    def __init__(self, rows): self.iloc=rows

def flags(model): return {n:p.requires_grad for n,p in model.named_parameters()}

def snapshot(x):
    if isinstance(x,torch.Tensor): return x.detach().clone()
    if isinstance(x,dict):return {k:snapshot(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)):return tuple(snapshot(v) for v in x)
    return x

def same(a,e):
    if isinstance(e,torch.Tensor):
        assert isinstance(a,torch.Tensor), 'Tensor를 반환하세요.'
        torch.testing.assert_close(a,e,rtol=2e-5,atol=2e-6)
    elif isinstance(e,dict):
        assert a.keys()==e.keys()
        for k in e:same(a[k],e[k])
    elif isinstance(e,tuple):
        assert len(a)==len(e)
        for x,y in zip(a,e):same(x,y)
    elif isinstance(e,float):assert abs(a-e)<1e-6,(a,e)
    else:assert a==e,(a,e)


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask', 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape 
        keys = self.W_key(x)        # Shape: [b, num_tokens, d_out]
        queries = self.W_query(x)   # Shape: [b, num_tokens, d_out]
        values = self.W_value(x)    # Shape: [b, num_tokens, d_out]
        attn_scores = queries @ keys.transpose(1, 2) 
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], 
            -torch.inf
        ) 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "출력 차원(d_out)은 헤드 수(num_heads)로 나누어 떨어져야 합니다."
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 각 헤드가 담당할 차원 크기
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out) 
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, num_tokens, d_in = x.shape # b: 배치 크기, num_tokens: 시퀀스 길이
        keys = self.W_key(x)     # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)  # 결과 Shape: (b, num_heads, num_tokens, num_tokens)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) 
        return context_vec

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim)) # 학습 가능한 스케일 파라미터 (Gamma)
        self.shift = nn.Parameter(torch.zeros(emb_dim)) # 학습 가능한 시프트 파라미터 (Beta)
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )
    def forward(self, x):
        return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        shortcut = x
        x = self.norm1(x) # Pre-LayerNorm 방식
        x = self.att(x)   
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 더해줌 (기울기 소실 방지)
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력 다시 더함
        return x

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), 
        target_batch.flatten()
    )
    return loss

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha
        self.rank = rank
    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )
    def forward(self, x):
        return self.linear(x) + self.lora(x)

def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

def observe(actual, kind, case, context):
    if kind=='optimizer':
        model=context['model']
        assert {id(p) for g in actual.param_groups for p in g['params']} == {id(p) for p in model.parameters()}
        assert isinstance(actual,torch.optim.AdamW)
        return actual.param_groups[0]['lr'],actual.param_groups[0]['weight_decay']
    if kind=='loss':
        assert actual.ndim==0
        actual.backward()
        gradients=tuple(p.grad for p in context['model'].parameters())
        assert any(g is not None and g.abs().sum()>0 for g in gradients)
        return snapshot((actual,gradients))
    if kind=='classification':
        m,fs=actual
        assert m.out_head.out_features==context['num_classes']
        assert all(not p.requires_grad for p in m.emb.parameters())
        assert all(not p.requires_grad for p in m.trf_blocks[0].parameters())
        assert all(p.requires_grad for p in m.trf_blocks[-1].parameters())
        assert all(p.requires_grad for p in m.final_norm.parameters())
        assert all(p.requires_grad for p in m.out_head.parameters())
        return snapshot((m(torch.tensor([[1,2],[3,4]])), fs, tuple(m.out_head.weight.shape)))
    if kind in ('attention','norm','ff','block','gpt','lora','wrapped','replace','freeze'):
        m=actual
        m.eval()
        if kind=='attention':
            x=torch.randn(case+1,4,context['d_in'],requires_grad=True)
        elif kind=='norm':
            with torch.no_grad():
                m.scale.copy_(torch.arange(context['emb_dim'])+.5)
                m.shift.copy_(torch.arange(context['emb_dim'])-.7)
            x=torch.randn(2,3,context['emb_dim'],requires_grad=True)
        elif kind in ('ff','block'):x=torch.randn(2,3,context['cfg']['emb_dim'],requires_grad=True)
        elif kind=='gpt':x=torch.tensor([[1,2,3],[3,2,1]])
        else:
            dim=context['in_dim'] if kind=='lora' else 3
            x=torch.randn(2,3,dim,requires_grad=True)
            with torch.no_grad():
                for name,p in m.named_parameters():
                    if name=='B' or name.endswith('.B'):p.copy_(torch.arange(p.numel()).reshape_as(p)*.05+.1)
        y=m(x)
        if kind=='attention':
            z=x.detach().clone();z[:,-1]+=10
            torch.testing.assert_close(m(z)[:,:-1],y[:,:-1],rtol=1e-5,atol=1e-6)
            assert 'mask' in dict(m.named_buffers())
            q=m.W_query(x);k=m.W_key(x);v=m.W_value(x)
            expected=torch.nn.functional.scaled_dot_product_attention(q,k,v,is_causal=True)
            torch.testing.assert_close(y,expected,rtol=1e-5,atol=1e-6)
        if kind=='norm':
            expected=torch.nn.functional.layer_norm(x,(context['emb_dim'],),m.scale,m.shift,m.eps)
            torch.testing.assert_close(y,expected,rtol=1e-5,atol=1e-6)
        y.square().mean().backward()
        grads=tuple((name,None if p.grad is None else p.grad.detach().clone()) for name,p in m.named_parameters())
        if kind=='freeze':
            for name,p in m.named_parameters():
                assert p.requires_grad == (name.endswith('.A') or name.endswith('.B')),name
        return snapshot((y,flags(m),grads))
    return snapshot(actual)


## 01. 토큰 인코딩·디코딩

원본: `Chapter_2_Exercise_Dataset.ipynb` **전체 셀 1** (Markdown 포함, 1부터), 원본 줄 16~24.

**목표:** 원본 빈칸을 완성하세요. 반환값: `(integers, strings)`.

**제공 입력/인자 예시:**
```python
case=0
tokenizer=DummyTokenizer()
text=['Hello, tea?', 'AI 공부'][case]
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_01(tokenizer, text):
    integers = tokenizer.MISSING(text, allowed_special={"<|endoftext|>"})  # ???? 원본 빈칸
    strings = tokenizer.MISSING(integers)  # ???? 원본 빈칸
    return (integers, strings)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_01():
    expected=(((72, 101, 108, 108, 111, 44, 32, 116, 101, 97, 63,), 'Hello, tea?',), ((65, 73, 32, 234, 179, 181, 235, 182, 128,), 'AI 공부',),)
    for case in range(2):
        torch.manual_seed(2010+case)
        tokenizer=DummyTokenizer()
        text=['Hello, tea?', 'AI 공부'][case]
        actual=solve_01(tokenizer, text)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 01 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_01()
except Exception as error:
    print('FAIL · 01 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

encode는 문자열→정수 목록, decode는 정수 목록→문자열입니다. 예: 더미에서 A→[65]→A. 실제 GPT-2 BPE 토큰 값과는 다릅니다.

```python
def solve_01(tokenizer, text):
    integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    strings = tokenizer.decode(integers)
    return (integers, strings)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 02. 다음 토큰 정답 한 칸 이동

원본: `Chapter_2_Exercise_Dataset.ipynb` **전체 셀 2** (Markdown 포함, 1부터), 원본 줄 30~31.

**목표:** 원본 빈칸을 완성하세요. 반환값: `(x,y)`.

**제공 입력/인자 예시:**
```python
case=0
enc_sample=list(range(10,22))
context_size=case+3
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_02(enc_sample, context_size):
    x = enc_sample[:context_size]
    y = enc_sample[MISSING:context_size+MISSING]  # ???? 원본 빈칸
    return (x,y)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_02():
    expected=(((10, 11, 12,), (11, 12, 13,),), ((10, 11, 12, 13,), (11, 12, 13, 14,),),)
    for case in range(2):
        torch.manual_seed(2020+case)
        enc_sample=list(range(10,22))
        context_size=case+3
        actual=solve_02(enc_sample, context_size)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 02 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_02()
except Exception as error:
    print('FAIL · 02 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

입력 [a,b,c]의 정답은 [b,c,d]. 끝 인덱스는 포함하지 않으므로 시작·끝 모두1씩 이동합니다.

```python
def solve_02(enc_sample, context_size):
    x = enc_sample[:context_size]
    y = enc_sample[1:context_size+1]
    return (x,y)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 03. GPTDataset 슬라이딩 윈도우

원본: `Chapter_2_Exercise_Dataset.ipynb` **전체 셀 3** (Markdown 포함, 1부터), 원본 줄 6~60.

**목표:** 원본 빈칸을 완성하세요. 반환값: `tuple(GPTDatasetV1(txt, tokenizer, max_length, stride))`.

**제공 입력/인자 예시:**
```python
case=0
txt='abcdefghijk'
tokenizer=DummyTokenizer()
max_length=case+3
stride=case+1
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_03(txt, tokenizer, max_length, stride):
    class GPTDatasetV1(Dataset):
        def __init__(self, txt, tokenizer, max_length, stride):
            self.input_ids = []
            self.target_ids = []
            token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
            assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."
            for i in range(0, len(token_ids) - max_length, stride):
                input_chunk = token_ids[MISSING : MISSING + max_length]  # ???? 원본 빈칸
                target_chunk = token_ids[MISSING + MISSING : MISSING + max_length + MISSING]  # ???? 원본 빈칸
                self.input_ids.append(torch.tensor(input_chunk))
                self.target_ids.append(torch.tensor(target_chunk))
        def __len__(self):
            return len(self.input_ids)
        def __getitem__(self, idx):
            return self.input_ids[MISSING], self.target_ids[MISSING]  # ???? 원본 빈칸
    return tuple(GPTDatasetV1(txt, tokenizer, max_length, stride))

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_03():
    expected=(((torch.tensor([97, 98, 99],dtype=torch.int64), torch.tensor([98, 99, 100],dtype=torch.int64),), (torch.tensor([98, 99, 100],dtype=torch.int64), torch.tensor([99, 100, 101],dtype=torch.int64),), (torch.tensor([99, 100, 101],dtype=torch.int64), torch.tensor([100, 101, 102],dtype=torch.int64),), (torch.tensor([100, 101, 102],dtype=torch.int64), torch.tensor([101, 102, 103],dtype=torch.int64),), (torch.tensor([101, 102, 103],dtype=torch.int64), torch.tensor([102, 103, 104],dtype=torch.int64),), (torch.tensor([102, 103, 104],dtype=torch.int64), torch.tensor([103, 104, 105],dtype=torch.int64),), (torch.tensor([103, 104, 105],dtype=torch.int64), torch.tensor([104, 105, 106],dtype=torch.int64),), (torch.tensor([104, 105, 106],dtype=torch.int64), torch.tensor([105, 106, 107],dtype=torch.int64),),), ((torch.tensor([97, 98, 99, 100],dtype=torch.int64), torch.tensor([98, 99, 100, 101],dtype=torch.int64),), (torch.tensor([99, 100, 101, 102],dtype=torch.int64), torch.tensor([100, 101, 102, 103],dtype=torch.int64),), (torch.tensor([101, 102, 103, 104],dtype=torch.int64), torch.tensor([102, 103, 104, 105],dtype=torch.int64),), (torch.tensor([103, 104, 105, 106],dtype=torch.int64), torch.tensor([104, 105, 106, 107],dtype=torch.int64),),),)
    for case in range(2):
        torch.manual_seed(2030+case)
        txt='abcdefghijk'
        tokenizer=DummyTokenizer()
        max_length=case+3
        stride=case+1
        actual=solve_03(txt, tokenizer, max_length, stride)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 03 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_03()
except Exception as error:
    print('FAIL · 03 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

입력 token_ids[i:i+L],정답 token_ids[i+1:i+L+1]. 각 샘플은 long 텐서 두 개, shape=(L,). stride로 시작 위치를 이동합니다.

```python
def solve_03(txt, tokenizer, max_length, stride):
    class GPTDatasetV1(Dataset):
        def __init__(self, txt, tokenizer, max_length, stride):
            self.input_ids = []
            self.target_ids = []
            token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
            assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."
            for i in range(0, len(token_ids) - max_length, stride):
                input_chunk = token_ids[i : i + max_length]
                target_chunk = token_ids[i + 1 : i + max_length + 1]
                self.input_ids.append(torch.tensor(input_chunk))
                self.target_ids.append(torch.tensor(target_chunk))
        def __len__(self):
            return len(self.input_ids)
        def __getitem__(self, idx):
            return self.input_ids[idx], self.target_ids[idx]
    return tuple(GPTDatasetV1(txt, tokenizer, max_length, stride))
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 04. DataLoader에 Dataset 전달

원본: `Chapter_2_Exercise_Dataset.ipynb` **전체 셀 4** (Markdown 포함, 1부터), 원본 줄 18~26.

**목표:** 원본 빈칸을 완성하세요. 반환값: `tuple(dataloader)`.

**제공 입력/인자 예시:**
```python
case=0
dataset=TensorDataset(torch.arange(15).reshape(5,3),torch.arange(5))
batch_size=case+2
shuffle=False
drop_last=bool(case)
num_workers=0
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_04(dataset, batch_size, shuffle, drop_last, num_workers):
    dataloader = DataLoader(
        MISSING,  # ???? 원본 빈칸
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return tuple(dataloader)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_04():
    expected=(((torch.tensor([[0, 1, 2], [3, 4, 5]],dtype=torch.int64), torch.tensor([0, 1],dtype=torch.int64),), (torch.tensor([[6, 7, 8], [9, 10, 11]],dtype=torch.int64), torch.tensor([2, 3],dtype=torch.int64),), (torch.tensor([[12, 13, 14]],dtype=torch.int64), torch.tensor([4],dtype=torch.int64),),), ((torch.tensor([[0, 1, 2], [3, 4, 5], [6, 7, 8]],dtype=torch.int64), torch.tensor([0, 1, 2],dtype=torch.int64),),),)
    for case in range(2):
        torch.manual_seed(2040+case)
        dataset=TensorDataset(torch.arange(15).reshape(5,3),torch.arange(5))
        batch_size=case+2
        shuffle=False
        drop_last=bool(case)
        num_workers=0
        actual=solve_04(dataset, batch_size, shuffle, drop_last, num_workers)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 04 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_04()
except Exception as error:
    print('FAIL · 04 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

DataLoader의 첫 인자는 Dataset입니다. 5샘플,batch2,drop_last=False → 배치크기2,2,1. 문제에서는 토큰화·파일 준비를 생략합니다.

```python
def solve_04(dataset, batch_size, shuffle, drop_last, num_workers):
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return tuple(dataloader)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 05. 토큰 임베딩 조회

원본: `Chapter_2_Exercise_Dataset.ipynb` **전체 셀 6** (Markdown 포함, 1부터), 원본 줄 11~20.

**목표:** 원본 빈칸을 완성하세요. 반환값: `(embedding_layer.weight, token_embeddings)`.

**제공 입력/인자 예시:**
```python
case=0
vocab_size=8+case
output_dim=3+case
input_ids=torch.tensor([2,0,5,2])
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_05(vocab_size, output_dim, input_ids):
    embedding_layer = torch.nn.Embedding(MISSING, MISSING)  # ???? 원본 빈칸
    token_embeddings = embedding_layer(MISSING)  # ???? 원본 빈칸
    return (embedding_layer.weight, token_embeddings)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_05():
    expected=((torch.tensor([[0.7221728563308716, 1.3032821416854858, -1.0118290185928345], [1.1299622058868408, -0.7002451419830322, 0.9815988540649414], [-1.5150251388549805, -1.5346665382385254, 0.9310657978057861], [0.6896814107894897, 0.4145013988018036, -1.0226361751556396], [-1.0762205123901367, 1.0086971521377563, -1.0481064319610596], [0.596778929233551, 0.15234169363975525, -1.1000089645385742], [-0.38056039810180664, -0.15024159848690033, 0.2894643247127533], [-2.449256420135498, -0.15628007054328918, -0.3175279498100281]],dtype=torch.float32), torch.tensor([[-1.5150251388549805, -1.5346665382385254, 0.9310657978057861], [0.7221728563308716, 1.3032821416854858, -1.0118290185928345], [0.596778929233551, 0.15234169363975525, -1.1000089645385742], [-1.5150251388549805, -1.5346665382385254, 0.9310657978057861]],dtype=torch.float32),), (torch.tensor([[0.02938711829483509, 0.028711026534438133, 0.45311641693115234, 0.5390235185623169], [0.1617918759584427, -0.4691309332847595, -1.0033735036849976, 0.14578475058078766], [0.21867971122264862, -0.2844025194644928, -1.8355844020843506, 2.30623197555542], [0.7697821855545044, -0.5998603701591492, -1.048044204711914, -0.7489873170852661], [-0.5077915787696838, 0.2492234706878662, -0.5964207053184509, -0.27937936782836914], [-0.40744879841804504, 0.14411216974258423, -0.5332990884780884, 0.48095420002937317], [-1.7601995468139648, -0.8018864989280701, -2.2218751907348633, 0.5131604671478271], [-1.7178637981414795, -0.10561876744031906, 0.9692711234092712, 0.2723028063774109], [-0.4904707372188568, -0.1439727544784546, 0.18801888823509216, 0.14455504715442657]],dtype=torch.float32), torch.tensor([[0.21867971122264862, -0.2844025194644928, -1.8355844020843506, 2.30623197555542], [0.02938711829483509, 0.028711026534438133, 0.45311641693115234, 0.5390235185623169], [-0.40744879841804504, 0.14411216974258423, -0.5332990884780884, 0.48095420002937317], [0.21867971122264862, -0.2844025194644928, -1.8355844020843506, 2.30623197555542]],dtype=torch.float32),),)
    for case in range(2):
        torch.manual_seed(2050+case)
        vocab_size=8+case
        output_dim=3+case
        input_ids=torch.tensor([2,0,5,2])
        actual=solve_05(vocab_size, output_dim, input_ids)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 05 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_05()
except Exception as error:
    print('FAIL · 05 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

Embedding(V,D)는 (V,D) 표입니다. ID4개를 넣으면 (4,D), 같은 ID는 같은 행을 꺼냅니다.

```python
def solve_05(vocab_size, output_dim, input_ids):
    embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
    token_embeddings = embedding_layer(input_ids)
    return (embedding_layer.weight, token_embeddings)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 06. CausalAttention 전체 흐름

원본: `Chapter_3_Excercise_Attention.ipynb` **전체 셀 1** (Markdown 포함, 1부터), 원본 줄 5~93.

**목표:** 원본 빈칸을 완성하세요. 반환값: `CausalAttention(d_in,d_out,context_length,0.0,qkv_bias)`.

**제공 입력/인자 예시:**
```python
case=0
d_in=3+case
d_out=2+case
context_length=5
qkv_bias=bool(case)
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_06(d_in, d_out, context_length, qkv_bias):
    class CausalAttention(nn.Module):
        def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
            super().__init__()
            self.W_query = nn.MISSING(MISSING, MISSING, bias=qkv_bias)  # ???? 원본 빈칸
            self.W_key   = nn.MISSING(MISSING, MISSING, bias=qkv_bias)  # ???? 원본 빈칸
            self.W_value = nn.MISSING(MISSING, MISSING, bias=qkv_bias)  # ???? 원본 빈칸
            self.dropout = nn.Dropout(dropout)
            self.register_buffer(
                'mask', 
                torch.triu(torch.ones(context_length, context_length), diagonal=1)
            )
        def forward(self, x):
            b, num_tokens, d_in = x.shape 
            keys = self.MISSING(MISSING)        # Shape: [b, num_tokens, d_out]  # ???? 원본 빈칸
            queries = self.MISSING(MISSING)   # Shape: [b, num_tokens, d_out]  # ???? 원본 빈칸
            values = self.MISSING(MISSING)    # Shape: [b, num_tokens, d_out]  # ???? 원본 빈칸
            attn_scores = MISSING @ MISSING.transpose(1, 2)   # ???? 원본 빈칸
            attn_scores.masked_fill_(
                self.mask.bool()[:num_tokens, :num_tokens], 
                MISSING  # ???? 원본 빈칸
            ) 
            attn_weights = torch.MISSING(  # ???? 원본 빈칸
                attn_scores / keys.shape[-1]**0.5, dim=-1
            )
            attn_weights = self.dropout(attn_weights)
            context_vec = attn_weights @ MISSING  # ???? 원본 빈칸
            return context_vec
    return CausalAttention(d_in,d_out,context_length,0.0,qkv_bias)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_06():
    expected=((torch.tensor([[[-0.944442629814148, -0.6781624555587769], [-0.4970990717411041, -0.4080355763435364], [-0.31607741117477417, -0.19599951803684235], [-0.248067244887352, -0.2079305797815323]]],dtype=torch.float32), {'W_query.weight':True, 'W_key.weight':True, 'W_value.weight':True}, (('W_query.weight', torch.tensor([[-0.00788485910743475, 0.0031881583854556084, -0.002463092328980565], [-0.010228129103779793, -0.0035693952813744545, 0.0015578677412122488]],dtype=torch.float32),), ('W_key.weight', torch.tensor([[0.005250981543213129, 0.002797122346237302, -0.0031831837259233], [0.003004056867212057, 0.003793009789660573, -0.00596046494320035]],dtype=torch.float32),), ('W_value.weight', torch.tensor([[0.8588440418243408, 0.02107512205839157, 0.5148823261260986], [0.6343494653701782, 0.018259800970554352, 0.37792372703552246]],dtype=torch.float32),),),), (torch.tensor([[[1.1508898735046387, 0.9781787395477295, -0.7505863308906555], [0.261113703250885, 0.2146969437599182, -0.18821561336517334], [0.1027081310749054, -0.10220538079738617, -0.09503233432769775], [0.5976977348327637, -0.1800832748413086, -0.2043292075395584]], [[0.12250271439552307, 1.2771527767181396, -0.5085127949714661], [0.512900710105896, 0.9224120378494263, -0.4113834500312805], [0.38474327325820923, 0.830859363079071, -0.39720335602760315], [0.36903446912765503, 0.41182899475097656, -0.2987564504146576]]],dtype=torch.float32), {'W_query.weight':True, 'W_query.bias':True, 'W_key.weight':True, 'W_key.bias':True, 'W_value.weight':True, 'W_value.bias':True}, (('W_query.weight', torch.tensor([[-0.005946638528257608, -0.0087540652602911, 0.0005359429633244872, 0.0013844940112903714], [0.0032400500494986773, 0.007830756716430187, -0.004307216499000788, 0.004819951485842466], [0.007291372399777174, -0.004485036712139845, -0.012860344722867012, -0.027349960058927536]],dtype=torch.float32),), ('W_query.bias', torch.tensor([-0.006028409115970135, 0.013803690671920776, -0.01519797183573246],dtype=torch.float32),), ('W_key.weight', torch.tensor([[-0.00547031732276082, 0.007711663376539946, -0.016246356070041656, -0.008517465554177761], [-0.00628472538664937, 0.008894449099898338, -0.02069496363401413, 0.012599251233041286], [0.003878804389387369, -0.00672058155760169, 0.017084183171391487, -0.017407573759555817]],dtype=torch.float32),), ('W_key.bias', torch.tensor([-3.259629011154175e-09, -6.984919309616089e-10, -2.0954757928848267e-09],dtype=torch.float32),), ('W_value.weight', torch.tensor([[0.1647004336118698, 0.11439356207847595, 0.23325896263122559, -0.22561001777648926], [0.3488859534263611, 0.1314719319343567, 0.1478458195924759, -0.47959697246551514], [-0.1716301292181015, -0.0946141928434372, -0.13336490094661713, 0.23981206119060516]],dtype=torch.float32),), ('W_value.bias', torch.tensor([0.29179924726486206, 0.36273667216300964, -0.23783497512340546],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2060+case)
        d_in=3+case
        d_out=2+case
        context_length=5
        qkv_bias=bool(case)
        actual=solve_06(d_in, d_out, context_length, qkv_bias)
        observed=observe(actual,'attention',case,locals())
        same(observed,expected[case])
    print('PASS · 06 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_06()
except Exception as error:
    print('FAIL · 06 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

Q,K,V는(B,T,Dout), QKᵀ는(B,T,T). 미래점수=-inf → softmax 확률0 → AV는(B,T,Dout). dropout=0으로 검사합니다.

```python
def solve_06(d_in, d_out, context_length, qkv_bias):
    class CausalAttention(nn.Module):
        def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
            super().__init__()
            self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
            self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
            self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
            self.dropout = nn.Dropout(dropout)
            self.register_buffer(
                'mask', 
                torch.triu(torch.ones(context_length, context_length), diagonal=1)
            )
        def forward(self, x):
            b, num_tokens, d_in = x.shape 
            keys = self.W_key(x)        # Shape: [b, num_tokens, d_out]
            queries = self.W_query(x)   # Shape: [b, num_tokens, d_out]
            values = self.W_value(x)    # Shape: [b, num_tokens, d_out]
            attn_scores = queries @ keys.transpose(1, 2) 
            attn_scores.masked_fill_(
                self.mask.bool()[:num_tokens, :num_tokens], 
                -torch.inf
            ) 
            attn_weights = torch.softmax(
                attn_scores / keys.shape[-1]**0.5, dim=-1
            )
            attn_weights = self.dropout(attn_weights)
            context_vec = attn_weights @ values
            return context_vec
    return CausalAttention(d_in,d_out,context_length,0.0,qkv_bias)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 07. Attention 인스턴스 호출

원본: `Chapter_3_Excercise_Attention.ipynb` **전체 셀 2** (Markdown 포함, 1부터), 원본 줄 28~28.

**목표:** 원본 빈칸을 완성하세요. 반환값: `context_vecs`.

**제공 입력/인자 예시:**
```python
case=0
ca=CausalAttention(3,2,6,0.0)
batch=torch.randn(case+1,4,3)
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_07(ca, batch):
    context_vecs = MISSING(batch)  # ???? 원본 빈칸
    return context_vecs

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_07():
    expected=(torch.tensor([[[0.6898254156112671, -0.23298095166683197], [-0.15160328149795532, 0.06282490491867065], [-0.35256046056747437, 0.2090153694152832], [-0.4853212833404541, 0.16694775223731995]]],dtype=torch.float32), torch.tensor([[[-0.1926400363445282, 0.46556317806243896], [-0.2507668733596802, 0.4716009199619293], [-0.039034292101860046, 0.12584485113620758], [-0.03966953232884407, 0.073751300573349]], [[-0.3364631235599518, 0.3615851402282715], [-0.4830171465873718, 0.8017019033432007], [-0.3661002218723297, 0.4871160089969635], [-0.2521660625934601, 0.24928681552410126]]],dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(2070+case)
        ca=CausalAttention(3,2,6,0.0)
        batch=torch.randn(case+1,4,3)
        actual=solve_07(ca, batch)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 07 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_07()
except Exception as error:
    print('FAIL · 07 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

ca(batch)는 __call__을 통해 forward를 실행합니다. (B,4,3) → (B,4,2).

```python
def solve_07(ca, batch):
    context_vecs = ca(batch)
    return context_vecs
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 08. LayerNorm 평균·분산·학습 파라미터

원본: `Chapter_4_Excercise_GPT.ipynb` **전체 셀 3** (Markdown 포함, 1부터), 원본 줄 5~23.

**목표:** 원본 빈칸을 완성하세요. 반환값: `LayerNorm(emb_dim)`.

**제공 입력/인자 예시:**
```python
case=0
emb_dim=case+3
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_08(emb_dim):
    class LayerNorm(nn.Module):
        def __init__(self, emb_dim):
            super().__init__()
            self.eps = 1e-5
            self.scale = nn.Parameter(torch.ones(emb_dim)) # 학습 가능한 스케일 파라미터 (Gamma)
            self.shift = nn.Parameter(torch.zeros(emb_dim)) # 학습 가능한 시프트 파라미터 (Beta)
        def forward(self, x):
            mean = x.mean(dim=-1, keepdim=True)
            var = x.var(dim=-1, keepdim=True, unbiased=False)
            norm_x = (x - MISSING) / torch.sqrt(MISSING + self.eps)  # ???? 원본 빈칸
            return self.MISSING * norm_x + self.MISSING  # ???? 원본 빈칸
    return LayerNorm(emb_dim)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_08():
    expected=((torch.tensor([[[-0.7749835848808289, 2.2392191886901855, -1.5571143627166748], [-0.024377644062042236, -0.17141225934028625, -1.2924249172210693], [-0.7052133083343506, 2.1442673206329346, -1.7477118968963623]], [[-0.21343162655830383, -1.7628681659698486, 2.305271625518799], [-0.8769630789756775, 2.3440847396850586, -1.2219927310943604], [-0.8665944337844849, -1.2354991436004639, 4.692137241363525]]],dtype=torch.float32), {'scale':True, 'shift':True}, (('scale', torch.tensor([0.05356184020638466, 1.385401964187622, 1.5307352542877197],dtype=torch.float32),), ('shift', torch.tensor([-0.38461819291114807, 0.39531025290489197, 0.13090723752975464],dtype=torch.float32),),),), (torch.tensor([[[-1.2667266130447388, 1.494471788406372, 4.236472129821777, -0.6310760974884033], [-1.0500662326812744, 1.9241580963134766, 3.481523036956787, -2.093370199203491], [-0.19282430410385132, -1.5758960247039795, 3.6713266372680664, -0.19299602508544922]], [[-0.38629984855651855, 2.032593011856079, 0.533336877822876, -2.8652899265289307], [-1.4136683940887451, 2.278085231781006, 0.6109441518783569, 3.644824981689453], [0.05289274454116821, 0.5052480697631836, 0.31439197063446045, -2.069310426712036]]],dtype=torch.float32), {'scale':True, 'shift':True}, (('scale', torch.tensor([0.31920260190963745, 0.8887811899185181, 0.9200497269630432, 0.9588330388069153],dtype=torch.float32),), ('shift', torch.tensor([-0.35472437739372253, 0.5548883676528931, 1.0706663131713867, -0.3506014943122864],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2080+case)
        emb_dim=case+3
        actual=solve_08(emb_dim)
        observed=observe(actual,'norm',case,locals())
        same(observed,expected[case])
    print('PASS · 08 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_08()
except Exception as error:
    print('FAIL · 08 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

각 토큰의 마지막 축에서 (x-mean)/sqrt(var+eps), 이후 scale×정규화+shift. 예: [1,3]의 평균2,분산1. 학습 scale/shift도 검사합니다.

```python
def solve_08(emb_dim):
    class LayerNorm(nn.Module):
        def __init__(self, emb_dim):
            super().__init__()
            self.eps = 1e-5
            self.scale = nn.Parameter(torch.ones(emb_dim)) # 학습 가능한 스케일 파라미터 (Gamma)
            self.shift = nn.Parameter(torch.zeros(emb_dim)) # 학습 가능한 시프트 파라미터 (Beta)
        def forward(self, x):
            mean = x.mean(dim=-1, keepdim=True)
            var = x.var(dim=-1, keepdim=True, unbiased=False)
            norm_x = (x - mean) / torch.sqrt(var + self.eps)
            return self.scale * norm_x + self.shift
    return LayerNorm(emb_dim)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 09. FFN 확장과 GELU

원본: `Chapter_4_Excercise_GPT.ipynb` **전체 셀 3** (Markdown 포함, 1부터), 원본 줄 40~57.

**목표:** 원본 빈칸을 완성하세요. 반환값: `FeedForward(cfg)`.

**제공 입력/인자 예시:**
```python
case=0
cfg={'emb_dim':case+3}
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_09(cfg):
    class FeedForward(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(cfg["emb_dim"], MISSING * cfg["emb_dim"]),  # ???? 원본 빈칸
                MISSING(),  # ???? 원본 빈칸
                nn.Linear(MISSING * cfg["emb_dim"], cfg["emb_dim"]),  # ???? 원본 빈칸
            )
        def forward(self, x):
            return self.layers(x)
    return FeedForward(cfg)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_09():
    expected=((torch.tensor([[[0.13077190518379211, -0.1286071538925171, -0.37203115224838257], [-0.04272790253162384, -0.15340785682201385, 0.030718162655830383], [0.21110770106315613, -0.16748958826065063, -0.09798862785100937]], [[-0.2457105815410614, -0.05812963843345642, 0.46087586879730225], [0.1552552878856659, -0.1478654146194458, -0.2030200958251953], [0.5928293466567993, 0.029155105352401733, -0.6021277904510498]]],dtype=torch.float32), {'layers.0.weight':True, 'layers.0.bias':True, 'layers.2.weight':True, 'layers.2.bias':True}, (('layers.0.weight', torch.tensor([[-0.010229172185063362, -0.0010945383692160249, -0.0027075435500591993], [0.02372026816010475, 0.01301314402371645, 0.005548984743654728], [0.011347571387887001, 0.019497975707054138, 0.0017559807747602463], [0.014436237514019012, 0.03559054434299469, 0.002665317850187421], [0.017446301877498627, 0.03693843632936478, -0.0024182554334402084], [-0.0868678018450737, -0.02911936491727829, -0.017586205154657364], [-0.007111602928489447, -0.002628448884934187, -0.0012580414768308401], [0.012585076503455639, 0.003764192573726177, 0.0031638878863304853], [-0.004343725740909576, 0.0004443085636012256, -0.001677996595390141], [0.060128867626190186, 0.04162983596324921, 0.011242726817727089], [-0.007383092772215605, -0.004224433097988367, -0.0008812141022644937], [-0.10839232057332993, -0.08023592084646225, -0.027162514626979828]],dtype=torch.float32),), ('layers.0.bias', torch.tensor([0.005117133259773254, -0.02084306627511978, 0.009501509368419647, 0.014544757083058357, 0.01928679272532463, 0.03291785717010498, 0.002081287559121847, -0.005319012328982353, -0.004631904419511557, -0.008484570309519768, -0.00406357366591692, 0.040589261800050735],dtype=torch.float32),), ('layers.2.weight', torch.tensor([[0.06578891724348068, 0.044572796672582626, -0.031043026596307755, -0.037970591336488724, -0.0473223440349102, 0.06550737470388412, 0.04021338373422623, 0.05552615225315094, -0.0063367560505867004, -0.0018459372222423553, -0.012829241342842579, 0.06877383589744568], [0.008540204726159573, 0.005572359077632427, 0.0032969010062515736, -0.008477026596665382, -0.012476014904677868, 0.013282331638038158, 0.011825806461274624, 0.01128820888698101, -0.00826200656592846, -0.0034736173693090677, -0.0029316642321646214, -0.01262945868074894], [-0.06861180812120438, -0.05273589491844177, 0.04471612349152565, 0.05992070958018303, 0.08324817568063736, -0.06641397625207901, -0.03810190409421921, -0.06018231436610222, -0.008150078356266022, 0.026841947808861732, 0.020737595856189728, -0.06687551736831665]],dtype=torch.float32),), ('layers.2.bias', torch.tensor([0.08905841410160065, -0.06959383934736252, -0.0870637521147728],dtype=torch.float32),),),), (torch.tensor([[[0.9061686396598816, 0.15794764459133148, 0.06152579188346863, 0.32654520869255066], [0.03412418067455292, 0.028705082833766937, 0.1721605360507965, 0.022110480815172195], [0.06609626114368439, 0.13424761593341827, 0.15011811256408691, 0.24035750329494476]], [[0.5008829832077026, 0.09044523537158966, 0.2159196436405182, 0.517868161201477], [0.26458704471588135, 0.024596892297267914, 0.15355660021305084, 0.35395798087120056], [0.07570328563451767, -0.0910714864730835, 0.0243774875998497, 0.03673933818936348]]],dtype=torch.float32), {'layers.0.weight':True, 'layers.0.bias':True, 'layers.2.weight':True, 'layers.2.bias':True}, (('layers.0.weight', torch.tensor([[-0.029238272458314896, 0.06096889078617096, -0.05485083535313606, 0.052299853414297104], [-0.0004711767833214253, -0.005031447857618332, 0.0019641730468720198, 0.0046337442472577095], [0.005179986357688904, -0.0028990046121180058, 0.004186893347650766, -0.002889297902584076], [0.0009829869959503412, -0.0039432537741959095, 0.0022933760192245245, -0.0016135683981701732], [-0.004441889934241772, 0.0015494839753955603, -0.0045664007775485516, 0.0024492815136909485], [0.0005201385356485844, 0.0026228546630591154, -0.0012800981057807803, -0.0007438157335855067], [-0.015808874741196632, 0.04147850722074509, -0.032584283500909805, 0.030449561774730682], [-0.004792878404259682, 0.010565320029854774, -0.007669731508940458, 0.009904772974550724], [-0.0014060763642191887, 0.0012361682020127773, 0.005682835355401039, 0.00939846970140934], [-0.0008759165648370981, -0.0006579735781997442, -0.0007463180227205157, 0.0002660188765730709], [0.0006410490022972226, 0.0004107528948225081, -9.17611105251126e-05, -0.0006174371810629964], [0.008487595245242119, -0.01974010281264782, 0.019936855882406235, -0.018333828076720238], [-0.026293594390153885, 0.038703832775354385, -0.04021000862121582, 0.047878239303827286], [0.0017435752088204026, -0.0015935980482026935, 3.680140434880741e-05, -0.009543545544147491], [0.01888471096754074, -0.052988287061452866, 0.04111570492386818, -0.03259354457259178], [0.00025930965784937143, -0.006461401470005512, 0.005579593125730753, 0.002113275695592165]],dtype=torch.float32),), ('layers.0.bias', torch.tensor([0.04566280543804169, 0.005973526276648045, 0.0014800329227000475, -0.002179669914767146, -0.003240520367398858, -0.0008736466988921165, 0.02842932753264904, 0.00859894324094057, 0.009076573885977268, -0.001772389281541109, 0.001244227634742856, -0.024532299488782883, 0.04309163987636566, -0.011592825874686241, -0.029483702033758163, 0.0021761185489594936],dtype=torch.float32),), ('layers.2.weight', torch.tensor([[0.10582050681114197, 0.006441024132072926, -0.018852895125746727, -0.023274481296539307, -0.023484546691179276, -0.005444364622235298, 0.15137144923210144, 0.13063298165798187, -0.0020354415755718946, -0.02063472755253315, -0.014824436977505684, 0.001381943468004465, 0.19406016170978546, 0.17058628797531128, 0.024185586720705032, 0.01217198371887207], [0.01818525232374668, -0.0023860845249146223, -0.0008814090979285538, -0.004732462111860514, -0.0054261027835309505, -0.00027841838891617954, 0.026936311274766922, 0.02343069016933441, 0.009579909965395927, -0.004151836037635803, -0.0051986840553581715, -0.0035500379744917154, 0.03262528032064438, 0.030784793198108673, 0.003586952108889818, 0.0022608237341046333], [0.013920233584940434, 0.005232851020991802, 0.0026138757821172476, -0.008646506816148758, -0.006407379638403654, -0.005013203248381615, 0.02386009506881237, 0.015321601182222366, 0.020526964217424393, -0.0021762156393378973, 0.006583536509424448, -0.0014244316844269633, 0.035348281264305115, 0.0338096022605896, -0.006011872552335262, 0.006169119384139776], [0.05342734977602959, 0.004817713517695665, -0.010912393219769001, -0.01931123435497284, -0.019139286130666733, -0.006501644384115934, 0.08198926597833633, 0.06349028646945953, 0.01871807500720024, -0.01715283840894699, -0.014008663594722748, -0.004123260732740164, 0.114961177110672, 0.10127011686563492, -0.0010348360519856215, 0.01848086714744568]],dtype=torch.float32),), ('layers.2.bias', torch.tensor([0.15396353602409363, 0.02873924747109413, 0.06480485200881958, 0.12479822337627411],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2090+case)
        cfg={'emb_dim':case+3}
        actual=solve_09(cfg)
        observed=observe(actual,'ff',case,locals())
        same(observed,expected[case])
    print('PASS · 09 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_09()
except Exception as error:
    print('FAIL · 09 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

D→4D→GELU→D. 예: (2,3,4)→(2,3,16)→(2,3,4). 원본의 tanh 근사 GELU가 준비 셀에 제공됩니다.

```python
def solve_09(cfg):
    class FeedForward(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
                GELU(),
                nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
            )
        def forward(self, x):
            return self.layers(x)
    return FeedForward(cfg)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 10. Transformer 잔차 두 경로

원본: `Chapter_4_Excercise_GPT.ipynb` **전체 셀 4** (Markdown 포함, 1부터), 원본 줄 6~44.

**목표:** 원본 빈칸을 완성하세요. 반환값: `TransformerBlock(cfg)`.

**제공 입력/인자 예시:**
```python
case=0
cfg={'emb_dim':4+case*2,'vocab_size':9+case,'context_length':6,'n_heads':2,'drop_rate':0.,'qkv_bias':True,'n_layers':2}
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_10(cfg):
    class TransformerBlock(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.att = MultiHeadAttention(
                d_in=cfg["emb_dim"],
                d_out=cfg["emb_dim"],
                context_length=cfg["context_length"],
                num_heads=cfg["n_heads"],
                dropout=cfg["drop_rate"],
                qkv_bias=cfg["qkv_bias"])
            self.ff = FeedForward(cfg)
            self.norm1 = LayerNorm(cfg["emb_dim"])
            self.norm2 = LayerNorm(cfg["emb_dim"])
            self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        def forward(self, x):
            shortcut = x
            x = self.norm1(x) # Pre-LayerNorm 방식
            x = self.MISSING(x)     # ???? 원본 빈칸
            x = self.drop_shortcut(x)
            x = x + shortcut  # 원본 입력을 더해줌 (기울기 소실 방지)
            shortcut = x
            x = self.norm2(x)
            x = self.MISSING(x)  # ???? 원본 빈칸
            x = self.drop_shortcut(x)
            x = x + shortcut  # 원본 입력 다시 더함
            return x
    return TransformerBlock(cfg)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_10():
    expected=((torch.tensor([[[-0.6628224849700928, -0.9929307103157043, 0.634952187538147, -0.5492085814476013], [-0.7254803776741028, 0.20143571496009827, -1.1037647724151611, -0.5578305721282959], [-1.9000563621520996, -0.7127971649169922, -0.1301840841770172, 0.47859200835227966]], [[-0.11550602316856384, 1.286889672279358, 0.7026240229606628, 0.22919383645057678], [-1.134873867034912, 1.78892183303833, 1.1151915788650513, 2.0928707122802734], [-2.359278440475464, 1.6395597457885742, -0.3915966749191284, -1.217369794845581]]],dtype=torch.float32), {'att.W_query.weight':True, 'att.W_query.bias':True, 'att.W_key.weight':True, 'att.W_key.bias':True, 'att.W_value.weight':True, 'att.W_value.bias':True, 'att.out_proj.weight':True, 'att.out_proj.bias':True, 'ff.layers.0.weight':True, 'ff.layers.0.bias':True, 'ff.layers.2.weight':True, 'ff.layers.2.bias':True, 'norm1.scale':True, 'norm1.shift':True, 'norm2.scale':True, 'norm2.shift':True}, (('att.W_query.weight', torch.tensor([[-0.0012940382584929466, -0.005206822883337736, 0.004410322289913893, 0.0020905390847474337], [0.011508905328810215, -0.011778302490711212, 0.02469191886484623, -0.024422520771622658], [-0.002435332862660289, 0.009891179390251637, -0.0037442785687744617, -0.003711567958816886], [0.0017887490103021264, -0.007132539059966803, 0.0015646957326680422, 0.0037790946662425995]],dtype=torch.float32),), ('att.W_query.bias', torch.tensor([-0.004607644863426685, -0.023428315296769142, 0.006687242537736893, -0.004269854631274939],dtype=torch.float32),), ('att.W_key.weight', torch.tensor([[-0.00042906717862933874, 0.03299279138445854, -0.029034193605184555, -0.003529526060447097], [-0.009146525524556637, 0.018573233857750893, -0.020079558715224266, 0.010652853175997734], [-0.011709004640579224, -0.0029673969838768244, -0.0006077071884647012, 0.015284109860658646], [0.005539299920201302, 0.003150450298562646, -0.001431143842637539, -0.007258606608957052]],dtype=torch.float32),), ('att.W_key.bias', torch.tensor([-1.3969838619232178e-09, -2.6775524020195007e-09, -3.4924596548080444e-10, 2.3283064365386963e-10],dtype=torch.float32),), ('att.W_value.weight', torch.tensor([[-0.018958043307065964, 0.23540954291820526, -0.09659925848245621, -0.11985225230455399], [-0.1005401611328125, 0.10907745361328125, -0.04498762637376785, 0.03645031899213791], [-0.11511993408203125, 0.22263625264167786, -0.11251511424779892, 0.004998791962862015], [0.06646264344453812, -0.18747910857200623, 0.10388718545436859, 0.017129307612776756]],dtype=torch.float32),), ('att.W_value.bias', torch.tensor([-0.06589102745056152, 0.19720105826854706, 0.3304957449436188, -0.14745527505874634],dtype=torch.float32),), ('att.out_proj.weight', torch.tensor([[0.1329679787158966, 0.015507772564888, -0.07174664735794067, -0.022977083921432495], [0.20950259268283844, 0.10505247116088867, 0.0760723203420639, 0.09267722070217133], [0.03806210681796074, 0.021979017183184624, 0.028684571385383606, -0.0036758417263627052], [0.08138216286897659, 0.03706762194633484, 0.0061828624457120895, 0.01336750015616417]],dtype=torch.float32),), ('att.out_proj.bias', torch.tensor([-0.6284509897232056, 0.2976117730140686, 0.1679438203573227, -0.03572699427604675],dtype=torch.float32),), ('ff.layers.0.weight', torch.tensor([[-0.009563447907567024, 0.012500958517193794, 0.02514190971851349, -0.028079424053430557], [-0.13158853352069855, 0.025250688195228577, 0.08403244614601135, 0.022305376827716827], [-0.03579173609614372, 0.1240171492099762, -0.02852514013648033, -0.05970027297735214], [-0.005216124001890421, 0.024566572159528732, -0.010232646949589252, -0.00911780260503292], [-0.0318654403090477, 0.032450251281261444, -0.0014148909831419587, 0.0008300743065774441], [-0.006670851726084948, 0.005187210626900196, -0.0005159117863513529, 0.0019995521288365126], [-0.04112856462597847, 0.014897421002388, 0.028867604210972786, -0.0026364577934145927], [-0.03770901635289192, 0.0076991720125079155, 0.009784984402358532, 0.020224859938025475], [-0.002904503606259823, 0.013474781066179276, 0.004953828640282154, -0.015524107962846756], [-0.04307761788368225, 0.030826492235064507, 0.010972918011248112, 0.0012782071717083454], [-0.013002312742173672, -0.0023435482289642096, 0.004205257631838322, 0.011140603572130203], [-0.04573572427034378, 0.03274612873792648, 0.00025793310487642884, 0.012731657363474369], [-0.1252034604549408, 0.09210948646068573, 0.011281332932412624, 0.021812643855810165], [-0.22075383365154266, 0.2420041859149933, -0.0006489634979516268, -0.020601412281394005], [0.08256970345973969, -0.06645287573337555, -0.00973738543689251, -0.006379431113600731], [0.013483579270541668, -0.001674590865150094, -0.01427893154323101, 0.002469942206516862]],dtype=torch.float32),), ('ff.layers.0.bias', torch.tensor([0.031827863305807114, 0.09901519864797592, 0.07125253230333328, 0.014210144989192486, 0.023551180958747864, 0.005377424415200949, 0.04567673057317734, 0.022786809131503105, 0.012019649147987366, 0.03242512419819832, 0.005189386662095785, 0.03031856380403042, 0.09891504794359207, 0.18703524768352509, -0.060494814068078995, -0.015248168259859085],dtype=torch.float32),), ('ff.layers.2.weight', torch.tensor([[0.0796472579240799, -0.4645773768424988, -0.11262188106775284, 0.058148469775915146, 0.04799460619688034, 0.0907818153500557, -0.05703665316104889, -0.4070931375026703, 0.07173006981611252, 0.06154696270823479, 0.0723871961236, 0.02319278009235859, -0.4149814546108246, -0.5922060608863831, -0.0387163870036602, 0.08087640255689621], [-0.05729655921459198, 0.16500437259674072, 0.14855150878429413, -0.017329776659607887, -0.04343005642294884, -0.0435669831931591, 1.0553162610449363e-05, 0.3803761601448059, -0.039835263043642044, -0.023929640650749207, -0.02842854894697666, -0.029390206560492516, 0.2129674106836319, 0.3535318672657013, 0.03665192425251007, -0.045697588473558426], [-0.0013961686054244637, 0.11526302993297577, -0.012401309795677662, -0.006558789871633053, -0.0008901707478798926, -0.014873081818223, 0.021687058731913567, -0.04294056445360184, -0.011739630252122879, -0.012194556184113026, -0.013009878806769848, -0.0014735754812136292, 0.05524125322699547, 0.09422808140516281, -0.004885073751211166, -0.00921264011412859], [-0.017023812979459763, 0.10222454369068146, -0.03330376371741295, -0.02515009231865406, 0.017256537452340126, -0.007067258935421705, 0.003236272605136037, -0.04144379496574402, -0.017065489664673805, -0.009728698991239071, -0.004819891881197691, 0.037968993186950684, 0.1687721610069275, 0.023548481985926628, 0.019770115613937378, 0.0011713526910170913]],dtype=torch.float32),), ('ff.layers.2.bias', torch.tensor([-0.5748348236083984, 0.2675899267196655, 0.06893519312143326, 0.03968731313943863],dtype=torch.float32),), ('norm1.scale', torch.tensor([0.0006885742768645287, 0.033767398446798325, 0.05238243192434311, 0.055131472647190094],dtype=torch.float32),), ('norm1.shift', torch.tensor([0.030626751482486725, 0.039034608751535416, 0.05802540481090546, 0.030896490439772606],dtype=torch.float32),), ('norm2.scale', torch.tensor([0.18319351971149445, 0.11270041763782501, 0.049571216106414795, 0.018523862585425377],dtype=torch.float32),), ('norm2.shift', torch.tensor([-0.13812290132045746, 0.09780801832675934, 0.08565346151590347, -0.06822527199983597],dtype=torch.float32),),),), (torch.tensor([[[0.8292601108551025, 1.8201582431793213, -0.19221431016921997, -0.852659285068512, -0.623038649559021, 0.6687624454498291], [0.3242838978767395, -0.4623430073261261, 0.16939884424209595, -1.4287478923797607, -1.6746186017990112, 0.316764771938324], [1.846338152885437, 0.057397231459617615, -0.07889096438884735, 0.625548243522644, 0.4679998755455017, -1.328026294708252]], [[0.7559813857078552, 0.06563636660575867, -0.7909835577011108, -0.8867470622062683, -1.6135010719299316, -0.3366876244544983], [2.1768062114715576, 0.10394251346588135, 1.0304678678512573, -1.313274621963501, 0.23605385422706604, 0.47599226236343384], [0.1687183976173401, 1.1009494066238403, 0.4011325240135193, -0.4793754816055298, -2.2052366733551025, 0.6738358736038208]]],dtype=torch.float32), {'att.W_query.weight':True, 'att.W_query.bias':True, 'att.W_key.weight':True, 'att.W_key.bias':True, 'att.W_value.weight':True, 'att.W_value.bias':True, 'att.out_proj.weight':True, 'att.out_proj.bias':True, 'ff.layers.0.weight':True, 'ff.layers.0.bias':True, 'ff.layers.2.weight':True, 'ff.layers.2.bias':True, 'norm1.scale':True, 'norm1.shift':True, 'norm2.scale':True, 'norm2.shift':True}, (('att.W_query.weight', torch.tensor([[0.026329217478632927, -0.015220042318105698, -0.010489127598702908, 0.017199989408254623, 0.023639939725399017, -0.04145997017621994], [0.02670329436659813, -0.005378958769142628, 0.011040500365197659, -0.030978146940469742, 0.0009595975279808044, -0.002346285618841648], [-0.021060064435005188, 0.007400861941277981, -0.0020180761348456144, 0.00979531928896904, -0.0080843735486269, 0.0139663340523839], [0.0029264995828270912, -0.000983693520538509, -0.0002933549985755235, 0.0018735621124505997, 0.0003122341586276889, -0.0038352468982338905], [0.01574813574552536, 0.0038544614799320698, 0.012377731502056122, -0.03186126425862312, -0.007696313783526421, 0.007577251642942429], [-0.015940843150019646, -0.0025447262451052666, -0.010298773646354675, 0.024079935625195503, 0.007289520930498838, -0.002585114212706685]],dtype=torch.float32),), ('att.W_query.bias', torch.tensor([0.02197522483766079, 0.024252478033304214, -0.018897635862231255, 9.417120600119233e-05, 0.022420842200517654, -0.017124520614743233],dtype=torch.float32),), ('att.W_key.weight', torch.tensor([[-0.011346719227731228, 0.007805400062352419, 0.00734863243997097, -0.021211208775639534, -0.007520944811403751, 0.02492484077811241], [-0.008223701268434525, -0.004111315123736858, 0.011713467538356781, -0.023956196382641792, 0.0019271622877568007, 0.0226505808532238], [0.0050859046168625355, -0.002576381666585803, -0.004119012504816055, 0.01030984241515398, 0.003231461625546217, -0.01193181425333023], [0.004271548241376877, -0.031420111656188965, 0.01783035695552826, -0.013399871066212654, 0.0174479428678751, 0.005270134191960096], [0.00908880215138197, -0.0666765421628952, 0.03618727624416351, -0.030710343271493912, 0.04234398901462555, 0.009766812436282635], [0.0037241086829453707, -0.026081133633852005, 0.015340890735387802, -0.009496183134615421, 0.012030370533466339, 0.00448194844648242]],dtype=torch.float32),), ('att.W_key.bias', torch.tensor([-4.838511813431978e-10, -1.0577423381619155e-09, 5.966285243630409e-10, 6.548361852765083e-11, -1.862645149230957e-09, -9.313225746154785e-10],dtype=torch.float32),), ('att.W_value.weight', torch.tensor([[0.09561057388782501, 0.13730111718177795, 0.00030024489387869835, -0.18820390105247498, -0.08616247773170471, 0.04115442931652069], [-0.04645960032939911, 0.01799769140779972, -0.00925600528717041, 0.03847435116767883, -0.010055672377347946, 0.009299235418438911], [0.06135905534029007, 0.0781659185886383, -0.0030912847723811865, -0.10119275748729706, -0.051616184413433075, 0.016375243663787842], [-0.05470157414674759, -0.09899614751338959, 0.004235161002725363, 0.09844093024730682, 0.08013010770082474, -0.029108460992574692], [-0.0730118378996849, -0.013131360523402691, 0.00218887347728014, 0.06975467503070831, 0.0013558026403188705, 0.012843858450651169], [0.10602171719074249, 0.11482646316289902, 0.007743622176349163, -0.19315290451049805, -0.07680927217006683, 0.04137036204338074]],dtype=torch.float32),), ('att.W_value.bias', torch.tensor([0.15379413962364197, -0.027110625058412552, 0.10214467346668243, -0.0952482745051384, -0.06308707594871521, 0.14666980504989624],dtype=torch.float32),), ('att.out_proj.weight', torch.tensor([[0.13174889981746674, 0.020823044702410698, 0.1839948296546936, 0.07036130130290985, -0.18484780192375183, -0.2037079930305481], [0.07149811834096909, -0.013163565658032894, 0.15510006248950958, 0.04563148319721222, -0.08314000070095062, -0.13387785851955414], [-0.00457722507417202, 0.01776091754436493, -0.006923386827111244, -0.03433035686612129, -0.0276021808385849, 0.03855413571000099], [-0.11025596410036087, 0.0055250730365514755, -0.1670042723417282, -0.03646381199359894, 0.1448875516653061, 0.14053381979465485], [-0.17312277853488922, 0.07282525300979614, -0.2793770432472229, -0.11787696182727814, 0.13998928666114807, 0.23742230236530304], [0.02560822293162346, -0.01176671963185072, 0.022970300167798996, -0.010875904932618141, -0.030855491757392883, -0.0059337373822927475]],dtype=torch.float32),), ('att.out_proj.bias', torch.tensor([0.3039524555206299, 0.14916911721229553, 0.01977640390396118, -0.20428967475891113, -0.2846496105194092, 0.018768131732940674],dtype=torch.float32),), ('ff.layers.0.weight', torch.tensor([[-0.0005573033122345805, 0.00949855800718069, 0.003146115690469742, -0.005209323950111866, -0.012095940299332142, 0.005217893980443478], [-0.007078700698912144, 0.03935037553310394, 0.001366548240184784, -0.021390173584222794, -0.04042284935712814, 0.028174791485071182], [0.01998310349881649, 0.010884198360145092, -0.002132860478013754, -0.018354026600718498, -0.002483080141246319, -0.00789733324199915], [0.04074123501777649, -0.04190543666481972, -0.0010669794864952564, 0.006646347232162952, 0.020111560821533203, -0.02452673390507698], [-0.0249997079372406, -0.017349354922771454, 0.004569218028336763, 0.0090691102668643, 0.01242300309240818, 0.016287731006741524], [0.027175838127732277, 0.010859529487788677, -0.007003981154412031, -0.00562638696283102, -0.007099124602973461, -0.01830587536096573], [0.006232274230569601, 0.002185660181567073, -0.0010202234843745828, -0.005504613276571035, 0.001081152237020433, -0.002974248956888914], [-0.002670591464266181, -0.01456691138446331, -0.0036253435537219048, 0.007986020296812057, 0.017862824723124504, -0.00498599698767066], [-0.010676558129489422, 0.002599093597382307, -0.0033090636134147644, 0.006728044711053371, -8.386338595300913e-05, 0.0047423457726836205], [-0.02959197573363781, 0.014206109568476677, 0.008761651813983917, -0.005414959974586964, -0.02346314862370491, 0.035502318292856216], [-0.05434331297874451, -0.0728914886713028, -0.01065003965049982, 0.08427424728870392, 0.07619652152061462, -0.022585922852158546], [0.017269356176257133, -0.007856614887714386, -0.006358536425977945, 0.020316842943429947, 0.006009337492287159, -0.02938038669526577], [-0.05179773271083832, -0.0166928730905056, -0.0002704877406358719, 0.07752714306116104, -0.013405542820692062, 0.0046394746750593185], [0.0373934730887413, 0.02232525125145912, -0.011182070709764957, -0.015658512711524963, -0.014986096881330013, -0.01789204403758049], [0.04108911752700806, 0.0366148017346859, -0.012949123978614807, -0.019398130476474762, -0.01958235539495945, -0.025774309411644936], [-0.003517830977216363, -0.0004991563619114459, -0.008405784144997597, 0.019789503887295723, 0.004930285271257162, -0.01229702029377222], [0.01449136808514595, 0.04602188616991043, -0.005314292851835489, -0.023691244423389435, -0.046139687299728394, 0.014631964266300201], [-0.021270601078867912, -0.022828303277492523, 0.0038876021280884743, 0.017764292657375336, 0.0218864344060421, 0.0005605794722214341], [-0.010753563605248928, 0.002082471502944827, 0.0044816406443715096, -0.0067313858307898045, -0.0030572335235774517, 0.013978072442114353], [-0.0018777975346893072, 0.015651382505893707, 0.0020035419147461653, -0.0078064026311039925, -0.013327190652489662, 0.005356467328965664], [0.049024056643247604, 0.024008046835660934, 0.008010639809072018, -0.05711536109447479, -0.027104122564196587, 0.0031767399050295353], [0.1073579490184784, 0.11290988326072693, 0.0314304381608963, -0.15012027323246002, -0.14818039536476135, 0.04660236835479736], [0.025754502043128014, -0.0044966815039515495, 0.005404501222074032, -0.0204519871622324, -0.00041830679401755333, -0.005792022682726383], [-0.046384140849113464, -0.06375977396965027, 0.003242450300604105, 0.05906887352466583, 0.05823291838169098, -0.0104003194719553]],dtype=torch.float32),), ('ff.layers.0.bias', torch.tensor([0.005811469163745642, 0.013577323406934738, 0.018798967823386192, -0.0018864208832383156, -0.026804184541106224, 0.022646604105830193, 0.005032931454479694, -0.011021499522030354, -0.007049767766147852, -0.012401444837450981, -0.07248727977275848, 0.010003289207816124, -0.03980979695916176, 0.03238649293780327, 0.04645393788814545, -0.002637373749166727, 0.027878183871507645, -0.022841021418571472, -0.006159605458378792, 0.008900073356926441, 0.043288059532642365, 0.1273012012243271, 0.01431192085146904, -0.05858731269836426],dtype=torch.float32),), ('ff.layers.2.weight', torch.tensor([[-0.03628166764974594, 0.011588425375521183, 0.018513670191168785, 0.11044731736183167, 0.080556720495224, -0.02408963069319725, -0.008247665129601955, -0.04940684139728546, 8.226111822295934e-05, 0.0205087848007679, 0.02231440134346485, 0.04329215735197067, 0.26103177666664124, 0.05875786393880844, 0.06498163193464279, 0.05238980054855347, -0.007416666951030493, -0.011717190034687519, -0.03341180086135864, -0.030789723619818687, 0.09365104138851166, 0.033560872077941895, 0.08801684528589249, 0.018297627568244934], [-0.02273242361843586, -0.019938252866268158, 0.05186958611011505, 0.10656174272298813, 0.0037136743776500225, -0.002665196545422077, -0.02411566488444805, -0.012749990448355675, -0.021549202501773834, 0.01173529401421547, 0.03621932491660118, -0.022049548104405403, 0.25936102867126465, -0.0031487380620092154, 0.04480393975973129, -0.02167483977973461, 0.030089860782027245, 0.02260986715555191, -0.021746711805462837, -0.008703002706170082, 0.04474135488271713, 0.06578848510980606, -0.01999724842607975, 0.026637079194188118], [-0.005254700779914856, 0.0173744335770607, -0.0012630005367100239, -0.025228019803762436, -0.010294473730027676, -0.012170884758234024, 0.01663055270910263, -0.00199722358956933, -0.005011463537812233, -0.001829112647101283, -5.794041862827726e-05, -0.007632601540535688, 0.024389412254095078, -0.01675451174378395, -0.02732689119875431, 0.0057265362702310085, -0.010585588403046131, -0.009475846774876118, -0.004896167665719986, -0.012649014592170715, 0.037008658051490784, 0.006127930246293545, -0.005106564145535231, -0.007691102102398872], [0.04714793339371681, -0.011855783872306347, -0.04207609221339226, -0.17470262944698334, 0.036924462765455246, 0.03171663358807564, 0.0020822363439947367, 0.03030407801270485, 0.053067415952682495, -0.03078174963593483, -0.07648636400699615, 0.07019208371639252, -0.407710999250412, 0.043880097568035126, 0.004697767551988363, 0.04866454377770424, -0.00699199503287673, 0.01684911549091339, 0.039264339953660965, 0.007739278022199869, -0.10421587526798248, -0.09192351251840591, 0.07476947456598282, -0.02469690702855587], [0.05239526554942131, 0.02399250864982605, -0.04997619614005089, -0.27515846490859985, 0.03636889532208443, 0.035737283527851105, 0.04118485748767853, 0.025186613202095032, 0.060890499502420425, -0.04198804497718811, -0.12016171216964722, 0.07461931556463242, -0.5919481515884399, 0.04544803500175476, -0.05487575754523277, 0.07303295284509659, -0.04912000149488449, -0.006497918628156185, 0.04839062690734863, -0.018516840413212776, -0.05687570199370384, -0.18418841063976288, 0.0825180932879448, -0.033764228224754333], [-0.0183341596275568, 0.006336516235023737, 0.03211202099919319, 0.05495796352624893, -0.0643596202135086, -0.011935959570109844, 0.00759316748008132, 0.0026420929934829473, -0.03910188004374504, 0.008672273717820644, 0.037534791976213455, -0.07281357795000076, 0.17690587043762207, -0.06326080858707428, -0.03761657699942589, -0.05500301718711853, 0.009564461186528206, -0.0013304674066603184, -0.014547467231750488, -0.0015648551052436233, 0.051449671387672424, 0.04972748085856438, -0.09514304250478745, 0.009866994805634022]],dtype=torch.float32),), ('ff.layers.2.bias', torch.tensor([0.3389660120010376, 0.14920783042907715, 0.029939468950033188, -0.24084757268428802, -0.3006856143474579, 0.026146739721298218],dtype=torch.float32),), ('norm1.scale', torch.tensor([0.01773795485496521, -0.012763869017362595, 0.009891879744827747, 0.17045561969280243, -0.015340207144618034, 0.02482299506664276],dtype=torch.float32),), ('norm1.shift', torch.tensor([0.011596797034144402, -0.00033177941804751754, 0.05787230283021927, -0.13285820186138153, 0.012466325424611568, 0.03965558111667633],dtype=torch.float32),), ('norm2.scale', torch.tensor([0.006379108875989914, 0.025488998740911484, 0.015731025487184525, -0.07890655845403671, -0.02637394890189171, 0.03755130246281624],dtype=torch.float32),), ('norm2.shift', torch.tensor([0.0027041705325245857, 0.016459766775369644, 0.0245131216943264, 0.07734178751707077, 0.04712105542421341, 0.01198544166982174],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2100+case)
        cfg={'emb_dim':4+case*2,'vocab_size':9+case,'context_length':6,'n_heads':2,'drop_rate':0.,'qkv_bias':True,'n_layers':2}
        actual=solve_10(cfg)
        observed=observe(actual,'block',case,locals())
        same(observed,expected[case])
    print('PASS · 10 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_10()
except Exception as error:
    print('FAIL · 10 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

LN→Attention→잔차합, LN→FFN→잔차합. 두 경로 모두 입력 shape 유지. MultiHeadAttention은 원본의 제공 코드를 사용합니다.

```python
def solve_10(cfg):
    class TransformerBlock(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.att = MultiHeadAttention(
                d_in=cfg["emb_dim"],
                d_out=cfg["emb_dim"],
                context_length=cfg["context_length"],
                num_heads=cfg["n_heads"],
                dropout=cfg["drop_rate"],
                qkv_bias=cfg["qkv_bias"])
            self.ff = FeedForward(cfg)
            self.norm1 = LayerNorm(cfg["emb_dim"])
            self.norm2 = LayerNorm(cfg["emb_dim"])
            self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        def forward(self, x):
            shortcut = x
            x = self.norm1(x) # Pre-LayerNorm 방식
            x = self.att(x)   
            x = self.drop_shortcut(x)
            x = x + shortcut  # 원본 입력을 더해줌 (기울기 소실 방지)
            shortcut = x
            x = self.norm2(x)
            x = self.ff(x)
            x = self.drop_shortcut(x)
            x = x + shortcut  # 원본 입력 다시 더함
            return x
    return TransformerBlock(cfg)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 11. GPT 임베딩·블록·출력층 조립

원본: `Chapter_4_Excercise_GPT.ipynb` **전체 셀 4** (Markdown 포함, 1부터), 원본 줄 45~95.

**목표:** 원본 빈칸을 완성하세요. 반환값: `GPTModel(cfg)`.

**제공 입력/인자 예시:**
```python
case=0
cfg={'emb_dim':4+case*2,'vocab_size':9+case,'context_length':6,'n_heads':2,'drop_rate':0.,'qkv_bias':True,'n_layers':2}
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_11(cfg):
    class GPTModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.tok_emb = nn.Embedding(cfg["MISSING"], cfg["emb_dim"])  # ???? 원본 빈칸
            self.pos_emb = nn.Embedding(cfg["MISSING"], cfg["emb_dim"])  # ???? 원본 빈칸
            self.drop_emb = nn.Dropout(cfg["drop_rate"])
            self.trf_blocks = nn.Sequential(
                *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
            self.final_norm = LayerNorm(cfg["emb_dim"])
            self.out_head = nn.Linear(cfg["emb_dim"], cfg["MISSING"], bias=False)  # ???? 원본 빈칸
        def forward(self, in_idx):
            batch_size, seq_len = in_idx.shape
            tok_embeds = self.tok_emb(in_idx)
            pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
            x = tok_embeds + MISSING  # ???? 원본 빈칸
            x = self.drop_emb(x)
            x = self.MISSING(x)  # ???? 원본 빈칸
            x = self.final_norm(x)
            logits = self.MISSING(x) # 각 단어에 대한 예측 점수 (Logits)  # ???? 원본 빈칸
            return logits
    return GPTModel(cfg)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_11():
    expected=((torch.tensor([[[-0.3311312198638916, -0.9800205230712891, -0.09710073471069336, -0.035039499402046204, -0.1200718879699707, 0.9323846697807312, -0.3459782600402832, 0.9226008653640747, -1.445436954498291], [0.047930896282196045, 0.24356931447982788, -0.2324668914079666, 0.8007205128669739, 0.929655909538269, 0.8692786693572998, -0.43818897008895874, 1.1632112264633179, -0.2366657555103302], [-0.06683672964572906, -0.8649311065673828, -0.11032244563102722, 0.9391424655914307, -0.696455180644989, 0.49273934960365295, -1.1195464134216309, 0.1613660752773285, -0.35663852095603943]], [[-0.34626880288124084, -0.764744222164154, 0.0364876389503479, -0.776136040687561, -0.18279705941677094, 0.4419974088668823, 0.30306994915008545, 0.4646450877189636, -1.3065015077590942], [0.02115611732006073, 0.2351902723312378, -0.22419434785842896, 0.6739583015441895, 0.971176028251648, 0.8861416578292847, -0.3385067880153656, 1.2082358598709106, -0.3352926969528198], [-0.02946285903453827, -0.6549530029296875, -0.1855851113796234, 1.162217617034912, -0.272568941116333, 0.7446287870407104, -1.1724969148635864, 0.5562399625778198, -0.3584308922290802]]],dtype=torch.float32), {'tok_emb.weight':True, 'pos_emb.weight':True, 'trf_blocks.0.att.W_query.weight':True, 'trf_blocks.0.att.W_query.bias':True, 'trf_blocks.0.att.W_key.weight':True, 'trf_blocks.0.att.W_key.bias':True, 'trf_blocks.0.att.W_value.weight':True, 'trf_blocks.0.att.W_value.bias':True, 'trf_blocks.0.att.out_proj.weight':True, 'trf_blocks.0.att.out_proj.bias':True, 'trf_blocks.0.ff.layers.0.weight':True, 'trf_blocks.0.ff.layers.0.bias':True, 'trf_blocks.0.ff.layers.2.weight':True, 'trf_blocks.0.ff.layers.2.bias':True, 'trf_blocks.0.norm1.scale':True, 'trf_blocks.0.norm1.shift':True, 'trf_blocks.0.norm2.scale':True, 'trf_blocks.0.norm2.shift':True, 'trf_blocks.1.att.W_query.weight':True, 'trf_blocks.1.att.W_query.bias':True, 'trf_blocks.1.att.W_key.weight':True, 'trf_blocks.1.att.W_key.bias':True, 'trf_blocks.1.att.W_value.weight':True, 'trf_blocks.1.att.W_value.bias':True, 'trf_blocks.1.att.out_proj.weight':True, 'trf_blocks.1.att.out_proj.bias':True, 'trf_blocks.1.ff.layers.0.weight':True, 'trf_blocks.1.ff.layers.0.bias':True, 'trf_blocks.1.ff.layers.2.weight':True, 'trf_blocks.1.ff.layers.2.bias':True, 'trf_blocks.1.norm1.scale':True, 'trf_blocks.1.norm1.shift':True, 'trf_blocks.1.norm2.scale':True, 'trf_blocks.1.norm2.shift':True, 'final_norm.scale':True, 'final_norm.shift':True, 'out_head.weight':True}, (('tok_emb.weight', torch.tensor([[0.0, 0.0, 0.0, 0.0], [0.007828455418348312, 0.004055849742144346, -0.004487012512981892, -0.007397288456559181], [0.00680414866656065, 0.01676987297832966, 0.0030834386125206947, -0.026657462120056152], [0.013270081952214241, 0.003302318276837468, -0.002628035843372345, -0.01394436415284872], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]],dtype=torch.float32),), ('pos_emb.weight', torch.tensor([[0.0064156875014305115, 0.01437382586300373, -0.01135649997740984, -0.009433010593056679], [0.00680414866656065, 0.01676987297832966, 0.0030834386125206947, -0.026657462120056152], [0.014682849869132042, -0.007015657611191273, 0.004241451155394316, -0.011908642016351223], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]],dtype=torch.float32),), ('trf_blocks.0.att.W_query.weight', torch.tensor([[-0.00022123876260593534, -0.00042138935532420874, 0.0004422169295139611, 0.00020041121752001345], [-0.0006044888868927956, 0.004050835035741329, -0.0015735598281025887, -0.001872786320745945], [-0.0019422699697315693, -0.00021284015383571386, 0.00201765401288867, 0.00013745628530159593], [0.0005224150372669101, 0.0001699947752058506, -0.0006013504462316632, -9.10594462766312e-05]],dtype=torch.float32),), ('trf_blocks.0.att.W_query.bias', torch.tensor([-0.00048171074013225734, 0.001835228642448783, -0.002113313414156437, 0.0006061053136363626],dtype=torch.float32),), ('trf_blocks.0.att.W_key.weight', torch.tensor([[0.0005747131071984768, -0.0002669345121830702, -0.0009690578444860876, 0.0006612791330553591], [0.0010682599386200309, -0.0010596197098493576, -0.0010052240686491132, 0.000996583723463118], [0.0008610229706391692, 0.0007245528977364302, -0.002965056337416172, 0.001379480236209929], [0.00020560156553983688, 0.0004898437764495611, -0.0009504650952294469, 0.00025501972413621843]],dtype=torch.float32),), ('trf_blocks.0.att.W_key.bias', torch.tensor([-4.729372449219227e-11, -1.4551915228366852e-11, 2.9103830456733704e-11, 1.4551915228366852e-11],dtype=torch.float32),), ('trf_blocks.0.att.W_value.weight', torch.tensor([[-0.0069396235048770905, 0.01016255747526884, -0.009221823886036873, 0.0059988913126289845], [0.005872596055269241, -0.0011558451224118471, -0.004693077877163887, -2.367363777011633e-05], [-0.007417209912091494, 0.006257779896259308, -0.0038730900269001722, 0.005032520741224289], [-0.01975822262465954, 0.024291781708598137, -0.023160135373473167, 0.018626581877470016]],dtype=torch.float32),), ('trf_blocks.0.att.W_value.bias', torch.tensor([-0.009435865096747875, 0.006858046166598797, -0.011241752654314041, -0.028991611674427986],dtype=torch.float32),), ('trf_blocks.0.att.out_proj.weight', torch.tensor([[0.0037619920913130045, -0.007441060617566109, -0.00017620268044993281, -0.002343324478715658], [0.0074074724689126015, -0.008929812349379063, -0.004354145843535662, -0.00024404923897236586], [-0.003109891666099429, 0.001061843940988183, 0.001059363828971982, 0.000599841121584177], [-0.008059574291110039, 0.015309029258787632, 0.003470985684543848, 0.001987532479688525]],dtype=torch.float32),), ('trf_blocks.0.att.out_proj.bias', torch.tensor([0.027393361553549767, 0.02201707661151886, -0.003252733498811722, -0.046157702803611755],dtype=torch.float32),), ('trf_blocks.0.ff.layers.0.weight', torch.tensor([[0.002625460736453533, -0.0016559967771172523, -0.0015873644733801484, 0.0006179008632898331], [0.0003632232255768031, -0.0003643610398285091, -0.00021845096489414573, 0.00021958873549010605], [0.00042167206993326545, -0.0010284635936841369, 0.002636282006278634, -0.0020294906571507454], [-0.0017977114766836166, 0.0009412404615432024, 0.0009741940884850919, -0.0001177230806206353], [-0.000813949853181839, 0.002648664638400078, -0.0041547296568751335, 0.002320015337318182], [-0.002868666546419263, 0.0027992501854896545, -0.0020886636339128017, 0.0021580797620117664], [-0.0004077617486473173, -0.0033424305729568005, -8.039665408432484e-05, 0.0038305888883769512], [0.0032484037801623344, -0.0042878263629972935, 0.0010018835309892893, 3.753961937036365e-05], [-2.2885389626026154e-05, 0.004433124326169491, -0.0029888288117945194, -0.0014214100083336234], [-0.008899529464542866, 0.006241116207093, 0.0032971478067338467, -0.0006387341418303549], [0.0016818067524582148, 0.0018115573329851031, 0.000578145612962544, -0.004071509465575218], [-0.00440774392336607, -0.00326099107041955, 0.001499612582847476, 0.0061691221781075], [0.008600111119449139, -0.004010980017483234, -0.0027661025524139404, -0.0018230290152132511], [-0.002135383663699031, 0.0009452573722228408, 0.0008432457689195871, 0.0003468805516604334], [-0.0021976041607558727, 0.0013400173047557473, 0.0010100639192387462, -0.00015247710689436644], [0.0026101807598024607, 0.00302500044927001, -0.0036369229201227427, -0.0019982578232884407]],dtype=torch.float32),), ('trf_blocks.0.ff.layers.0.bias', torch.tensor([0.0016356041887775064, -6.27480330877006e-05, 0.0016790764639154077, -0.0013026215601712465, -0.0019496651366353035, -0.0034206113778054714, -0.004078393802046776, 0.0020422684028744698, 0.0019900889601558447, -0.00666690431535244, 0.005083044525235891, -0.008852449245750904, 0.007771018892526627, -0.0019405294442549348, -0.0016020373441278934, 0.004374039825052023],dtype=torch.float32),), ('trf_blocks.0.ff.layers.2.weight', torch.tensor([[0.0007944334065541625, -0.003661802504211664, -0.0025336812250316143, -0.003674577223137021, 0.007741447072476149, 0.002822167705744505, 0.0009556394070386887, 0.005048247519880533, 0.00542550440877676, 0.001101998146623373, 0.004826352000236511, 0.0051299831829965115, 0.008642922155559063, 0.005981354974210262, -0.001730729709379375, 0.013903859071433544], [0.008453711867332458, -0.00287148030474782, -0.0007534205797128379, -0.0021245309617370367, -0.004078110679984093, 0.011243236251175404, -0.0013782999012619257, -0.004347308073192835, 0.0038827809039503336, 0.008704224601387978, -2.6581141355563886e-05, -0.002849301788955927, -0.0034975651651620865, 0.014124858193099499, 0.0008203522302210331, -0.0037170846480876207], [0.005044905934482813, 3.2575855584582314e-05, -0.001519876765087247, 0.00020866128033958375, 0.0002455732028465718, -0.009152974002063274, -0.0016526157269254327, 0.0012200003257021308, 0.004455240443348885, 0.00022587999410461634, -0.0033090661745518446, 0.0007405536016449332, 0.003456861013546586, 0.005203221924602985, 0.0021668802946805954, 0.008644717745482922], [-0.014293053187429905, 0.006500707473605871, 0.004806979093700647, 0.005590447224676609, -0.0039089093916118145, -0.0049124304205179214, 0.002075276803225279, -0.0019209393067285419, -0.013763527385890484, -0.010032104328274727, -0.0014907033182680607, -0.0030212339479476213, -0.008602217771112919, -0.02530943974852562, -0.0012565037468448281, -0.018831493332982063]],dtype=torch.float32),), ('trf_blocks.0.ff.layers.2.bias', torch.tensor([0.023868538439273834, 0.01814526692032814, -9.051291272044182e-06, -0.04200475662946701],dtype=torch.float32),), ('trf_blocks.0.norm1.scale', torch.tensor([-0.002755496185272932, -0.004307482857257128, -0.0023395575117319822, 0.0051346467807888985],dtype=torch.float32),), ('trf_blocks.0.norm1.shift', torch.tensor([-0.002214836422353983, 0.003706784453243017, -0.0016494451556354761, -0.005739286541938782],dtype=torch.float32),), ('trf_blocks.0.norm2.scale', torch.tensor([0.008626515977084637, -0.003099468071013689, 0.00015359290409833193, -0.0013029687106609344],dtype=torch.float32),), ('trf_blocks.0.norm2.shift', torch.tensor([0.010889868251979351, 0.010180407203733921, -0.0012559788301587105, -0.002584761707112193],dtype=torch.float32),), ('trf_blocks.1.att.W_query.weight', torch.tensor([[-0.0002184102195315063, 0.0009679803624749184, -0.0001701970468275249, -0.0005793731543235481], [-9.5157723990269e-05, 0.00011054069182137027, 3.117378582828678e-05, -4.655676821130328e-05], [0.000802778173238039, 0.0007725219475105405, -0.000789601996075362, -0.0007856981246732175], [-0.0016214840579777956, -0.0007147653959691525, 0.001351997023448348, 0.0009842521976679564]],dtype=torch.float32),), ('trf_blocks.1.att.W_query.bias', torch.tensor([0.0003334014327265322, -3.117551386822015e-05, 0.00134123838506639, -0.0021459630224853754],dtype=torch.float32),), ('trf_blocks.1.att.W_key.weight', torch.tensor([[0.0007952338200993836, -0.001977246953174472, 0.0003944962518289685, 0.0007875169394537807], [1.3279621271067299e-05, 0.00013139525253791362, -0.00020568117906805128, 6.10063252679538e-05], [-0.0017191452207043767, 0.001954480540007353, 0.0015478250570595264, -0.0017831606091931462], [-0.0007274436648003757, 0.002911921124905348, -0.0024775059428066015, 0.00029302865732461214]],dtype=torch.float32),), ('trf_blocks.1.att.W_key.bias', torch.tensor([-2.0372681319713593e-10, -4.547473508864641e-12, -4.220055416226387e-10, -4.220055416226387e-10],dtype=torch.float32),), ('trf_blocks.1.att.W_value.weight', torch.tensor([[-0.02326296456158161, 0.01592205837368965, -0.00346815912052989, 0.010809063911437988], [-0.027031129226088524, 0.021585019305348396, -0.008861524984240532, 0.014307638630270958], [-0.014993064105510712, 0.007667242083698511, -0.0004285639151930809, 0.007754385471343994], [-0.016300562769174576, 0.01830335520207882, -0.017796825617551804, 0.01579403504729271]],dtype=torch.float32),), ('trf_blocks.1.att.W_value.bias', torch.tensor([-0.023385005071759224, -0.027602938935160637, -0.01618899405002594, -0.019444307312369347],dtype=torch.float32),), ('trf_blocks.1.att.out_proj.weight', torch.tensor([[-0.00770785566419363, -0.004474441520869732, 0.017048025503754616, -0.017893237993121147], [-0.007551358547061682, 0.00041785358916968107, 0.010039363987743855, -0.01748717576265335], [-0.0011122676078230143, 0.0013223644345998764, 0.0014127582544460893, 0.006989698391407728], [0.016371482983231544, 0.002734222449362278, -0.02850014716386795, 0.02839071862399578]],dtype=torch.float32),), ('trf_blocks.1.att.out_proj.bias', torch.tensor([0.026627231389284134, 0.016875185072422028, -0.0014101609122008085, -0.04209225997328758],dtype=torch.float32),), ('trf_blocks.1.ff.layers.0.weight', torch.tensor([[0.00023708451772108674, -0.002493822481483221, 0.0036870120093226433, -0.001430274685844779], [0.00014204607577994466, 0.0020951698534190655, -0.0011100948322564363, -0.0011271208059042692], [-0.0005184123874641955, -0.004020607098937035, -0.00016850733663886786, 0.004707526881247759], [0.0017244573682546616, -0.002022044500336051, 0.00435552466660738, -0.004057937767356634], [0.007575388997793198, 0.001348651829175651, -0.005901404656469822, -0.003022635355591774], [0.001058737514540553, -0.001253864262253046, 0.0015889679780229926, -0.0013938413467258215], [-0.0006638398626819253, 0.0007256713579408824, -0.0018370471661910415, 0.001775215845555067], [-0.0010045371018350124, -0.0025046546943485737, 0.0022696703672409058, 0.0012395213125273585], [-0.0010845053475350142, 0.0009146584197878838, -0.0012435546377673745, 0.0014134019147604704], [-0.0007348422659561038, 0.00038410938577726483, 0.0012685165274888277, -0.0009177837055176497], [-4.568627628032118e-05, 0.0006569221732206643, -0.0003190289135091007, -0.0002922068815678358], [0.005374439992010593, -0.00033089134376496077, -0.004707395564764738, -0.00033615250140428543], [0.005577803589403629, -0.002592995297163725, -0.0006214362219907343, -0.0023633723612874746], [-0.0031478938180953264, 0.002588614821434021, 0.0019358627032488585, -0.0013765834737569094], [-0.002664653118699789, 0.0015517460415139794, 0.00029133365023881197, 0.0008215735433623195], [-0.0006584702059626579, 0.0029184671584516764, -0.00011653278488665819, -0.0021434640511870384]],dtype=torch.float32),), ('trf_blocks.1.ff.layers.0.bias', torch.tensor([-0.0002593938843347132, 0.0012824836885556579, -0.0033307885751128197, 0.0021296371705830097, 0.0073710596188902855, 0.0008998322882689536, -0.0012548260856419802, -0.0020219634752720594, -0.001174860168248415, -5.8117148000746965e-06, 0.0003096801810897887, 0.004084456246346235, 0.004529133904725313, -0.0010811274405568838, -0.0020808789413422346, 0.0015448452904820442],dtype=torch.float32),), ('trf_blocks.1.ff.layers.2.weight', torch.tensor([[0.010657249949872494, -0.0013240129919722676, 0.011344090104103088, 0.014428653754293919, 0.021590596064925194, 0.012570218183100224, -0.0010977387428283691, -0.002928662346675992, -0.003747466951608658, -0.002477453788742423, -0.004180894698947668, -0.0008219500305131078, 0.001952013117261231, 0.020761404186487198, -0.00335677876137197, -8.385206456296146e-05], [-0.0016304436139762402, -0.0025090754497796297, 0.010745037347078323, 0.004456725437194109, 0.003244709223508835, -0.00315711903385818, 0.0018203963991254568, -0.0024809688329696655, -0.0019605380948632956, -0.0017793825827538967, -0.0022335760295391083, 0.0018387660384178162, 0.007767566479742527, 0.006789264269173145, -0.0008696742006577551, -0.0021184561774134636], [0.0046037109568715096, -0.0002006827126024291, 0.003406594507396221, -0.006650862284004688, 0.0044480846263468266, -0.0010520125506445765, -0.004143979866057634, 0.0007763969479128718, -0.0004996646894142032, 0.00025806823396123946, -0.0001703334564808756, 0.0015567434020340443, -0.007139607332646847, 0.004536543507128954, 0.00011526237358339131, 0.0007923014345578849], [-0.013630516827106476, 0.004033771809190512, -0.025495724752545357, -0.012234514579176903, -0.029283389449119568, -0.008361084386706352, 0.0034213229082524776, 0.0046332343481481075, 0.006207670085132122, 0.003998768050223589, 0.006584804505109787, -0.0025735606905072927, -0.002579971682280302, -0.032087210565805435, 0.00411119032651186, 0.0014100067783147097]],dtype=torch.float32),), ('trf_blocks.1.ff.layers.2.bias', torch.tensor([0.02723022922873497, 0.015593787655234337, -0.0018262672238051891, -0.04099775105714798],dtype=torch.float32),), ('trf_blocks.1.norm1.scale', torch.tensor([-0.002369895111769438, -0.0024912438821047544, 0.0053891995921730995, 0.001405665883794427],dtype=torch.float32),), ('trf_blocks.1.norm1.shift', torch.tensor([-0.0019079797202721238, 0.007140078581869602, 0.012356551364064217, 0.0017418516799807549],dtype=torch.float32),), ('trf_blocks.1.norm2.scale', torch.tensor([0.006072649732232094, 0.0008392536547034979, -6.668613059446216e-05, 0.0005600496660917997],dtype=torch.float32),), ('trf_blocks.1.norm2.shift', torch.tensor([0.0036368239670991898, 0.00412304000928998, 0.0030829652678221464, -0.0005878577940165997],dtype=torch.float32),), ('final_norm.scale', torch.tensor([0.2827709913253784, 0.14910194277763367, 0.20605216920375824, 0.27198344469070435],dtype=torch.float32),), ('final_norm.shift', torch.tensor([0.26598820090293884, 0.0877360850572586, -0.07103604078292847, -0.2696633040904999],dtype=torch.float32),), ('out_head.weight', torch.tensor([[-0.01459205150604248, 0.00033530115615576506, -0.025041276589035988, 0.039298027753829956], [-0.035381633788347244, -0.07672353088855743, -0.034194186329841614, 0.146299347281456], [-0.03460875526070595, -0.009547818452119827, 0.02770005539059639, 0.016456516459584236], [0.10071185976266861, 0.10693354904651642, -0.16618071496486664, -0.04146469756960869], [0.1002105250954628, -0.08105801045894623, -0.060967665165662766, 0.041815146803855896], [0.16420546174049377, 0.03081020899116993, -0.06722380220890045, -0.1277918666601181], [-0.08113279938697815, -0.11963805556297302, 0.11039117723703384, 0.09037968516349792], [0.1968419998884201, -0.008411649614572525, -0.07581388205289841, -0.11261646449565887], [-0.11789177358150482, -0.004480078816413879, -0.05358216166496277, 0.17595401406288147]],dtype=torch.float32),),),), (torch.tensor([[[0.19541192054748535, 0.5834287405014038, 0.27832716703414917, 0.013465866446495056, 0.19864381849765778, 0.08056014031171799, -0.6735731959342957, -1.278039574623108, -0.07172027230262756, 0.33369261026382446], [0.5141908526420593, 0.47429758310317993, -0.9872249364852905, -0.22517934441566467, -0.8074455261230469, -1.2739694118499756, -0.9757950305938721, -0.34047725796699524, -0.6357989311218262, 1.038221836090088], [0.07859879732131958, 0.4718080461025238, -0.3872935175895691, 0.39858633279800415, -0.5404248833656311, -0.8903329372406006, -0.4696662724018097, -0.15123340487480164, -0.2719040811061859, 1.1752727031707764]], [[-0.391498863697052, -0.30293789505958557, -0.5851300358772278, -0.06386200338602066, 0.18987125158309937, -0.23705285787582397, -0.18837620317935944, -0.36771443486213684, 0.8547309041023254, 0.7844758629798889], [0.44420045614242554, 0.35383063554763794, -1.018418312072754, -0.20570144057273865, -0.7493090033531189, -1.219817042350769, -0.8474355340003967, -0.21868644654750824, -0.5283031463623047, 1.0062006711959839], [0.40156134963035583, 1.0728378295898438, 0.676770806312561, 0.6588181257247925, -0.249298095703125, -0.205864816904068, -0.48179876804351807, -0.7643637657165527, -0.862585723400116, 0.5207639932632446]]],dtype=torch.float32), {'tok_emb.weight':True, 'pos_emb.weight':True, 'trf_blocks.0.att.W_query.weight':True, 'trf_blocks.0.att.W_query.bias':True, 'trf_blocks.0.att.W_key.weight':True, 'trf_blocks.0.att.W_key.bias':True, 'trf_blocks.0.att.W_value.weight':True, 'trf_blocks.0.att.W_value.bias':True, 'trf_blocks.0.att.out_proj.weight':True, 'trf_blocks.0.att.out_proj.bias':True, 'trf_blocks.0.ff.layers.0.weight':True, 'trf_blocks.0.ff.layers.0.bias':True, 'trf_blocks.0.ff.layers.2.weight':True, 'trf_blocks.0.ff.layers.2.bias':True, 'trf_blocks.0.norm1.scale':True, 'trf_blocks.0.norm1.shift':True, 'trf_blocks.0.norm2.scale':True, 'trf_blocks.0.norm2.shift':True, 'trf_blocks.1.att.W_query.weight':True, 'trf_blocks.1.att.W_query.bias':True, 'trf_blocks.1.att.W_key.weight':True, 'trf_blocks.1.att.W_key.bias':True, 'trf_blocks.1.att.W_value.weight':True, 'trf_blocks.1.att.W_value.bias':True, 'trf_blocks.1.att.out_proj.weight':True, 'trf_blocks.1.att.out_proj.bias':True, 'trf_blocks.1.ff.layers.0.weight':True, 'trf_blocks.1.ff.layers.0.bias':True, 'trf_blocks.1.ff.layers.2.weight':True, 'trf_blocks.1.ff.layers.2.bias':True, 'trf_blocks.1.norm1.scale':True, 'trf_blocks.1.norm1.shift':True, 'trf_blocks.1.norm2.scale':True, 'trf_blocks.1.norm2.shift':True, 'final_norm.scale':True, 'final_norm.shift':True, 'out_head.weight':True}, (('tok_emb.weight', torch.tensor([[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.006696316413581371, -0.03535700589418411, 0.014334728009998798, 0.005590425338596106, 0.024121727794408798, -0.015386180952191353], [0.0027201841585338116, 0.007359425537288189, 0.03265877440571785, 0.011551032774150372, 0.0025352328084409237, -0.05682465061545372], [0.0027741561643779278, 0.010709119960665703, 0.009257606230676174, -0.008220899850130081, 0.008012309670448303, -0.022532299160957336], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]],dtype=torch.float32),), ('pos_emb.weight', torch.tensor([[-0.007957244291901588, 0.002062101848423481, -0.006828082725405693, -0.004427370149642229, 0.017937682569026947, -0.0007870884146541357], [0.0027201841585338116, 0.007359425537288189, 0.03265877440571785, 0.011551032774150372, 0.0025352328084409237, -0.05682465061545372], [0.0174277164041996, -0.026709988713264465, 0.030420416966080666, 0.0017968956381082535, 0.014196353033185005, -0.03713139146566391], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]],dtype=torch.float32),), ('trf_blocks.0.att.W_query.weight', torch.tensor([[0.006578363478183746, -0.0051409597508609295, 0.0072602336294949055, -0.004769479855895042, -0.003109222510829568, -0.0008189344662241638], [0.00988923292607069, -0.007751512806862593, 0.009568098932504654, -0.007155254483222961, -0.003699903842061758, -0.0008506610756739974], [0.00574070867151022, -0.004422512371093035, 0.006527753546833992, -0.004150927998125553, -0.0035065931733697653, -0.0001884288212750107], [-0.006222066003829241, 0.004843594040721655, -0.00648751063272357, 0.004501412622630596, 0.002941223094239831, 0.00042334606405347586], [-0.005042476579546928, 0.003928348422050476, -0.005470793694257736, 0.0036518231499940157, 0.002465762197971344, 0.00046733650378882885], [0.0024276438634842634, -0.001824449049308896, 0.0035498030483722687, -0.0017568779876455665, -0.0024021295830607414, 6.010173819959164e-06]],dtype=torch.float32),), ('trf_blocks.0.att.W_query.bias', torch.tensor([0.005667390767484903, 0.007575914263725281, 0.005020559765398502, -0.005069225560873747, -0.0042642648331820965, 0.002644728170707822],dtype=torch.float32),), ('trf_blocks.0.att.W_key.weight', torch.tensor([[0.002789088524878025, -0.002673821523785591, 0.0008480766555294394, 0.0007027198444120586, -0.0005647320067510009, -0.0011013310868293047], [-0.014183178544044495, 0.012734901160001755, -0.0025602257810533047, 0.0007318845018744469, 0.001906036282889545, 0.0013705827295780182], [0.0029076063074171543, -0.0022918223403394222, -8.496036753058434e-05, -0.0016775885596871376, -0.00018952242680825293, 0.0013362874742597342], [-0.011600205674767494, 0.010098356753587723, -0.0016127073904499412, 0.0019163262331858277, 0.0018482488812878728, -0.0006500199669972062], [0.010720870457589626, -0.009479648433625698, 0.001612306572496891, -0.0013236610684543848, -0.001181991770863533, -0.000347876048181206], [-0.0006698392098769546, 0.0005263940547592938, -8.445620915153995e-05, 0.0002204480697400868, 0.0004623024433385581, -0.00045484938891604543]],dtype=torch.float32),), ('trf_blocks.0.att.W_key.bias', torch.tensor([0.0, 5.093170329928398e-11, 1.1641532182693481e-10, 1.4551915228366852e-10, 0.0, -8.731149137020111e-11],dtype=torch.float32),), ('trf_blocks.0.att.W_value.weight', torch.tensor([[0.015411825850605965, -0.011134783737361431, -0.005575266666710377, -0.013852514326572418, 0.0016684560105204582, 0.013482283800840378], [0.002970116911455989, -0.0046326229348778725, -0.0236189067363739, 0.014317738823592663, 0.003779818071052432, 0.007183853071182966], [-0.007195534184575081, 0.0017970750341191888, -0.023979704827070236, 0.028419962152838707, 0.004186869133263826, -0.0032286718487739563], [0.009515670128166676, -0.006909104064106941, -0.001105590956285596, -0.008797907270491123, 0.0005625287303701043, 0.0067344047129154205], [0.0038637416437268257, -0.0018777505028992891, 0.016431402415037155, -0.01227585133165121, -0.000984331825748086, -0.0051572141237556934], [0.024310307577252388, -0.012750611640512943, 0.05241421237587929, -0.05773945525288582, -0.007829817943274975, 0.0015953618567436934]],dtype=torch.float32),), ('trf_blocks.0.att.W_value.bias', torch.tensor([-0.002181381219998002, -0.02197391539812088, -0.024933625012636185, 0.00045760232023894787, 0.015206065028905869, 0.052089933305978775],dtype=torch.float32),), ('trf_blocks.0.att.out_proj.weight', torch.tensor([[-0.003536198753863573, 0.013264620676636696, -0.011857504025101662, -0.013219999149441719, 0.010174239054322243, -0.020657461136579514], [0.007306354120373726, -0.0005187662318348885, 0.005802720319479704, 0.02473052591085434, -0.0036237621679902077, 0.008199513889849186], [-0.015392228029668331, 0.02606137841939926, -0.03516429290175438, -0.047528259456157684, 0.024721506983041763, -0.0490291491150856], [-0.005230652634054422, 0.002624588320031762, -0.0036119255237281322, -0.01195861492305994, 0.004053198266774416, -0.008342924527823925], [-0.005241423845291138, 0.005427121184766293, -0.017163116484880447, -0.021447790786623955, 0.006314066704362631, -0.011700283735990524], [0.02209414914250374, -0.04685894027352333, 0.06199411675333977, 0.06942413747310638, -0.04163924604654312, 0.08153028786182404]],dtype=torch.float32),), ('trf_blocks.0.att.out_proj.bias', torch.tensor([0.024302728474140167, -0.023299215361475945, 0.07164239138364792, 0.014148007147014141, 0.0271424762904644, -0.11393638700246811],dtype=torch.float32),), ('trf_blocks.0.ff.layers.0.weight', torch.tensor([[0.005332789849489927, -0.00509275309741497, 0.004354465752840042, -0.00416557677090168, -0.002629459137097001, 0.0022005338687449694], [-0.0022775577381253242, -0.000819497974589467, 0.01328408345580101, -0.0012479107826948166, -0.005989790894091129, -0.0029493258334696293], [0.010149475187063217, -0.009332401677966118, 0.007990235462784767, -0.008948937989771366, -0.003658157540485263, 0.0037997879553586245], [-0.002796344691887498, 0.002807814395055175, -0.002622456755489111, 0.0018839461263269186, 0.0004388429515529424, 0.00028819736326113343], [0.008465469814836979, -0.010025176219642162, 0.01367909088730812, -0.005814986303448677, -0.007430574391037226, 0.0011261763283982873], [0.0066059660166502, -0.008584852330386639, 0.013839298859238625, -0.004443835932761431, -0.006765863858163357, -0.0006507110665552318], [-0.0026989756152033806, 3.102072514593601e-05, 0.010353196412324905, -0.00010769686195999384, -0.004222007002681494, -0.003355539171025157], [-0.012747013941407204, 0.012072062119841576, -0.009866890497505665, 0.009136147797107697, 0.002505672862753272, -0.0010999813675880432], [-0.018953245133161545, 0.020666977390646935, -0.019567260518670082, 0.008708948269486427, 0.009073094464838505, 7.148273289203644e-05], [-0.00269290991127491, -0.0006296003703027964, 0.017680436372756958, -0.004679707810282707, -0.009348250925540924, -0.0003299661912024021], [-0.0032263384200632572, 0.00019081763457506895, 0.01386301126331091, -0.0019091737922281027, -0.004928707145154476, -0.003989609889686108], [-0.005695935804396868, 0.0036853691563010216, 0.005538949742913246, 0.0015060475561767817, -0.0025510501582175493, -0.0024833809584379196], [-0.0052795554511249065, 0.004517467692494392, -0.000992076937109232, 0.002770015737041831, -0.0005804242100566626, -0.00043542764615267515], [0.0005920735420659184, 0.000217500637518242, -0.0016563244862481952, -0.0014323460636660457, 0.0006089688977226615, 0.0016701278509572148], [0.006061997264623642, -0.0024477927945554256, -0.009292799979448318, -0.003915560897439718, 0.005449548363685608, 0.004144608974456787], [-0.008726089261472225, 0.008347881026566029, -0.00942706037312746, 0.009216388687491417, 0.0053687626495957375, -0.004779883660376072], [-0.00581866130232811, 0.005990080069750547, -0.006938238628208637, 0.00506505137309432, 0.0037892437539994717, -0.002087476197630167], [0.02466558665037155, -0.023132720962166786, 0.02016107179224491, -0.020181579515337944, -0.00720235425978899, 0.005689999554306269], [-0.001791553688235581, 0.001215397845953703, 6.237707566469908e-05, 0.0018665619427338243, 0.00016945770767051727, -0.0015222413931041956], [0.00024860945995897055, 0.0009350418113172054, -0.0033776341006159782, -0.0010639377869665623, 0.002471591578796506, 0.0007863293285481632], [-0.015157614834606647, 0.014293287880718708, -0.013565948233008385, 0.013872053474187851, 0.0069267177022993565, -0.0063684964552521706], [0.004506408702582121, -0.003356698201969266, -0.003016807371750474, -0.0008393081370741129, 0.0006564862560480833, 0.0020499187521636486], [0.012425011955201626, -0.010618695057928562, 0.0028666320722550154, -0.007476190570741892, -0.0019833191763609648, 0.004786562640219927], [0.00949022825807333, -0.012155176140367985, 0.01964118704199791, -0.007207890041172504, -0.011046991683542728, 0.0012786437291651964]],dtype=torch.float32),), ('trf_blocks.0.ff.layers.0.bias', torch.tensor([0.004302033688873053, 0.009539601393043995, 0.007979728281497955, -0.0020804931409657, 0.011067923158407211, 0.010754304006695747, 0.007091650273650885, -0.00895540602505207, -0.015347108244895935, 0.014185216277837753, 0.0099742216989398, 0.0036384002305567265, -0.0011165423784404993, -0.000622456893324852, -0.005695311352610588, -0.009267441928386688, -0.006263711489737034, 0.01908828690648079, -0.0005486331065185368, -0.0021039985585957766, -0.0132195008918643, -0.0018464000895619392, 0.00406613340601325, 0.01576167903840542],dtype=torch.float32),), ('trf_blocks.0.ff.layers.2.weight', torch.tensor([[0.008591385558247566, 0.022757386788725853, -0.00036392922629602253, -0.003225373337045312, 0.011809957213699818, 0.01363134291023016, 0.011852370575070381, 0.011261184699833393, 0.029478421434760094, 0.010825718753039837, 0.0013908246764913201, -0.003050662577152252, -0.002101338468492031, -0.004044198896735907, 0.008678690530359745, 0.00644037127494812, 0.01622907631099224, 0.002952929586172104, -0.003763720626011491, -0.001914899330586195, 0.006345310714095831, -0.003919054288417101, -0.0001357892615487799, 0.01720854826271534], [-0.01864011585712433, -0.02550836093723774, 0.01102457009255886, 0.008251826278865337, 0.0014784580562263727, -0.013656613416969776, 0.006176048889756203, 0.015143830329179764, 0.021412229165434837, -0.03271666541695595, -0.023132406175136566, -0.0006118317251093686, -0.017847005277872086, 0.004198082722723484, -0.014788130298256874, 0.005225106608122587, 0.008838162757456303, 0.009663663804531097, 0.00391753576695919, 0.00014082778943702579, 0.015823040157556534, 0.0021508350037038326, -0.004457281902432442, 0.009073660708963871], [0.022658061236143112, 0.048000484704971313, -0.003070219187065959, -0.008890096098184586, 0.021589254960417747, 0.02441132813692093, 0.023086711764335632, 0.025235004723072052, 0.06282178312540054, 0.026280531659722328, 0.009600630961358547, -0.0071761407889425755, 0.004920490086078644, -0.008308331482112408, 0.02654842846095562, 0.009786238893866539, 0.03301806375384331, 0.008453700691461563, -0.00892157107591629, -0.004518266301602125, 0.006974582560360432, -0.008210338652133942, 0.0015461049042642117, 0.037921320647001266], [0.004972279537469149, 0.012044761329889297, -0.0028733881190419197, -0.004194511100649834, 0.004863685928285122, 0.006334369536489248, 0.004220186732709408, 0.00867481715977192, 0.019248010590672493, 0.00815487653017044, 0.0040519824251532555, -0.0019446592777967453, 0.0013640354154631495, -0.004437322728335857, 0.011555397883057594, 0.0006716388743370771, 0.0073610045947134495, 0.0033478017430752516, -0.003042575903236866, -0.003989067859947681, -0.0030504781752824783, -0.0031328152399510145, -0.0004016113525722176, 0.01334855891764164], [0.019650744274258614, 0.02513817325234413, -0.0026479684747755527, -0.0030830560717731714, 0.0024852724745869637, 0.009365377947688103, 0.003509646514430642, -0.003940817900002003, -0.0014294872526079416, 0.023773394525051117, 0.015752773731946945, -0.0014736236771568656, 0.014633415266871452, 0.0001879640476545319, 0.012500500306487083, -0.002821424277499318, 0.004820309113711119, -0.0022106138058006763, -0.0039250850677490234, 0.003137551713734865, -0.0012488350039348006, -0.0015559385064989328, 0.005147454794496298, -0.0013242830755189061], [-0.03723234683275223, -0.08243244141340256, -0.0020690658129751682, 0.011141209863126278, -0.042226631194353104, -0.0400857999920845, -0.048844967037439346, -0.056374020874500275, -0.13153095543384552, -0.0363178476691246, -0.00766380038112402, 0.014256917871534824, -0.000969596323557198, 0.012403804808855057, -0.0444948785007, -0.019301937893033028, -0.07026661932468414, -0.022207483649253845, 0.01573541760444641, 0.007143852300941944, -0.024843623861670494, 0.014667308889329433, -0.001698876265436411, -0.0762278139591217]],dtype=torch.float32),), ('trf_blocks.0.ff.layers.2.bias', torch.tensor([0.027063904330134392, -0.025728296488523483, 0.05980943515896797, 0.017413567751646042, 0.027707325294613838, -0.10626592487096786],dtype=torch.float32),), ('trf_blocks.0.norm1.scale', torch.tensor([0.0006126615917310119, 0.0006645722314715385, -0.017964845523238182, 0.003862655023112893, -0.004427935462445021, 0.003108029020950198],dtype=torch.float32),), ('trf_blocks.0.norm1.shift', torch.tensor([-0.023645924404263496, 0.013055282644927502, -0.02051408216357231, -0.004051951691508293, 0.011268602684140205, 0.019830333068966866],dtype=torch.float32),), ('trf_blocks.0.norm2.scale', torch.tensor([-0.00435225386172533, -0.009592745453119278, 0.021730447188019753, 0.0015869645867496729, 0.0022948256228119135, -0.0017038355581462383],dtype=torch.float32),), ('trf_blocks.0.norm2.shift', torch.tensor([-0.007151630241423845, 0.006867446005344391, 0.01929846592247486, -0.0024443247821182013, 0.00019833468832075596, -0.009707184508442879],dtype=torch.float32),), ('trf_blocks.1.att.W_query.weight', torch.tensor([[0.00473773991689086, -0.006078043486922979, 0.0058686090633273125, -0.003037706483155489, -0.0023351600393652916, 0.0008445607963949442], [-0.004298062529414892, 0.00469471700489521, -0.003396133426576853, 0.003006732789799571, 0.0007780016749165952, -0.0007852562121115625], [-0.0005732140853069723, 0.0002470430335961282, 0.0007287433836609125, 0.00046827958431094885, -0.0004037090693600476, -0.0004671428760048002], [0.002173773944377899, -0.0022187423892319202, 0.0010015355655923486, -0.0015291024465113878, -0.00029989363974891603, 0.0008724291110411286], [0.00036073726369068027, 5.4982228903099895e-06, -0.0004921372747048736, -0.0004009075346402824, 0.0006534578278660774, -0.0001266483450308442], [0.0014572766376659274, -0.0016842626500874758, 0.0010549905709922314, -0.0009480765438638628, -0.0005499247345142066, 0.0006699966033920646]],dtype=torch.float32),), ('trf_blocks.1.att.W_query.bias', torch.tensor([0.005144636612385511, -0.003255600808188319, 0.0004035258025396615, 0.0011734407162293792, -0.0003124758368358016, 0.0010720681166276336],dtype=torch.float32),), ('trf_blocks.1.att.W_key.weight', torch.tensor([[-0.001709755975753069, 0.0012137165758758783, 0.0006685572443529963, 0.0012814318761229515, -0.00043736110092140734, -0.001016588881611824], [-0.01059204526245594, 0.009677128866314888, 0.0015309990849345922, 0.000760229304432869, -0.0003967570373788476, -0.0009795553050935268], [0.009238604456186295, -0.008486051112413406, -0.0014672819525003433, -0.0006773538189008832, 0.00010290733189322054, 0.0012891765218228102], [-0.00299768871627748, 0.002618118654936552, 0.001079695881344378, 0.0009612705325707793, 0.0002747768012341112, -0.0019361731829121709], [-0.0011939586838707328, 0.0008180774166248739, 0.0006931765237823129, 0.0009858813136816025, -0.00014637272397521883, -0.0011568039190024137], [0.00010476614988874644, -0.0001228339970111847, 2.2984946554061025e-05, 9.173857688438147e-05, -1.9409142623771913e-05, -7.724652823526412e-05]],dtype=torch.float32),), ('trf_blocks.1.att.W_key.bias', torch.tensor([-5.820766091346741e-11, -3.055902197957039e-10, 5.675246939063072e-10, -1.382431946694851e-10, -5.820766091346741e-11, 1.8189894035458565e-11],dtype=torch.float32),), ('trf_blocks.1.att.W_value.weight', torch.tensor([[-0.003739280393347144, 0.000837586005218327, 0.011248143389821053, -0.001097206142731011, -0.0018987328512594104, -0.005350510589778423], [0.009119762107729912, -0.00419234111905098, -0.015242773108184338, -0.0016583644319325686, 0.004242441616952419, 0.00773127656430006], [0.02285536751151085, -0.0228789821267128, 0.022637199610471725, -0.03185608237981796, -0.0021662190556526184, 0.01140871923416853], [-0.0029447637498378754, 0.0008634205441921949, 0.007680466398596764, -0.0019295683596283197, -0.0023736953735351562, -0.0012958594597876072], [-0.013725562021136284, 0.015089184045791626, -0.02194514311850071, 0.026057692244648933, 0.003246572334319353, -0.008722738362848759], [-0.00520573602989316, 0.011155669577419758, -0.03902805596590042, 0.027100522071123123, 0.005935221444815397, 4.237785469740629e-05]],dtype=torch.float32),), ('trf_blocks.1.att.W_value.bias', torch.tensor([0.006734346970915794, -0.008542145602405071, 0.02479070983827114, 0.005421245936304331, -0.022205151617527008, -0.031863268464803696],dtype=torch.float32),), ('trf_blocks.1.att.out_proj.weight', torch.tensor([[0.012471330352127552, -0.0006632122676819563, -0.01207322534173727, -0.00767377857118845, 0.008798087015748024, 0.0002465333673171699], [0.0030119498260319233, 0.01299169659614563, -0.009481801651418209, 0.01596941612660885, -0.0015308964066207409, 0.012527470476925373], [0.014276555739343166, -0.009239370934665203, -0.010016256012022495, -0.026164164766669273, 0.010398562066257, -0.014186921529471874], [0.004958163481205702, -0.004140615928918123, -0.004676803480833769, -0.009857574477791786, 0.0026920705568045378, -0.006099776830524206], [-0.002354865428060293, -0.005596883594989777, 0.007403340190649033, -0.007645968813449144, 0.0005439980886876583, -0.005796145182102919], [-0.032363127917051315, 0.006648385897278786, 0.028844740241765976, 0.03537207096815109, -0.020901817828416824, 0.013308844529092312]],dtype=torch.float32),), ('trf_blocks.1.att.out_proj.bias', torch.tensor([0.035093799233436584, -0.021992124617099762, 0.05561777204275131, 0.015437397174537182, 0.013825904577970505, -0.0979827344417572],dtype=torch.float32),), ('trf_blocks.1.ff.layers.0.weight', torch.tensor([[-0.0008807488484308124, -0.0006383154541254044, 0.0021442766301333904, 0.0016068199183791876, -0.0009396752575412393, -0.0012923573376610875], [0.007274376228451729, -0.00781534519046545, 0.0022352731321007013, -0.005989097990095615, 0.0007119562942534685, 0.003582838922739029], [-0.01341969147324562, 0.010793570429086685, 0.0031853830441832542, 0.007566733751446009, -0.0034881008323282003, -0.004637896083295345], [-0.001850051456131041, 0.0024373310152441263, -0.002324680332094431, 0.0008244980708695948, -0.000442129879957065, 0.0013550325529649854], [-0.002220926573500037, 0.005183928180485964, -0.00655113160610199, 0.0046628424897789955, 0.0014073658967390656, -0.00248207850381732], [-0.001810869900509715, -0.0009382246644236147, 0.005045083351433277, -0.00048332824371755123, -0.0019045276567339897, 9.186641545966268e-05], [-0.011548560112714767, 0.02169237658381462, -0.02037210948765278, 0.016123581677675247, 0.004897021222859621, -0.010792311280965805], [0.00298065016977489, -8.113868534564972e-05, -0.0071831680834293365, 0.0012886531185358763, 0.0016095397295430303, 0.0013854638673365116], [-0.0061225276440382, 0.008176957257091999, -0.004474652465432882, 0.004724208265542984, 0.00027693092124536633, -0.0025809179060161114], [0.0020968217868357897, -2.839200897142291e-05, -0.0027617011219263077, -0.0007132841856218874, 0.0018049232894554734, -0.00039836729411035776], [-0.00811689905822277, 0.01914849691092968, -0.02468198351562023, 0.015416956506669521, 0.00482166837900877, -0.0065882387571036816], [-0.0020810274872928858, 0.0018470720387995243, 0.0006535380962304771, 0.0010098735801875591, -0.00029283572803251445, -0.0011366207618266344], [-0.01685122586786747, 0.029543567448854446, -0.028650609776377678, 0.022781312465667725, 0.004690977744758129, -0.011514023877680302], [0.0069413622841238976, -0.004995521157979965, -0.002452367451041937, -0.005153561942279339, 0.001972086261957884, 0.003688002470880747], [6.601883796975017e-05, 0.0008127483306452632, -0.0007551567396149039, -0.0008235982968471944, 0.0007463255897164345, -4.6337678213603795e-05], [0.01218253094702959, -0.02024199441075325, 0.018370110541582108, -0.014300152659416199, -0.002528667449951172, 0.00651817349717021], [0.0007635678630322218, 0.00598384952172637, -0.01551411859691143, 0.006274599581956863, 0.0033889373298734426, -0.0008968361653387547], [0.0034701800905168056, -0.010056408122181892, 0.01356496848165989, -0.007089271675795317, -0.0032897561322897673, 0.003400287125259638], [-0.001098435721360147, 0.0008583372691646218, -0.00034310558112338185, 0.0008201058954000473, -0.0005229205708019435, 0.00028601859230548143], [0.0022819028235971928, -0.00668750237673521, 0.00808113906532526, -0.0032995131332427263, -0.002366873901337385, 0.0019908463582396507], [-0.003963540308177471, 0.005842819809913635, -0.0026954011991620064, 0.00520488852635026, 0.0013222541892901063, -0.005711020901799202], [0.0001633374486118555, -0.004772101063281298, 0.009074848145246506, -0.002773861400783062, -0.0025400076992809772, 0.0008477849187329412], [-0.00917801819741726, 0.004998223856091499, 0.007564930245280266, 0.002856919076293707, -0.003518562065437436, -0.002723494078963995], [-0.0020756153389811516, -0.004237120505422354, 0.012654194608330727, -0.0029748068191111088, -0.0036960174329578876, 0.00032936513889580965]],dtype=torch.float32),), ('trf_blocks.1.ff.layers.0.bias', torch.tensor([0.0006130158435553312, 0.002539073582738638, 0.0009578203316777945, -0.0010700130369514227, -0.0051338765770196915, 0.003216430312022567, -0.016052085906267166, -0.004653166513890028, -0.003477843478322029, -0.001736198551952839, -0.018325917422771454, 0.00019974798487965018, -0.022228658199310303, -0.00041292456444352865, 2.8651556931436062e-05, 0.013924832455813885, -0.011049487628042698, 0.009697155095636845, -0.0002506407618056983, 0.005409893114119768, -0.003445291193202138, 0.0060640424489974976, 0.004696665797382593, 0.008561204187572002],dtype=torch.float32),), ('trf_blocks.1.ff.layers.2.weight', torch.tensor([[0.003224854590371251, 0.0037883410695940256, 0.0018261588411405683, 0.011900384910404682, 0.017317429184913635, 0.006394443567842245, 0.015669073909521103, 0.006125343032181263, 0.0014337898464873433, -0.0020607535261660814, 0.023000717163085938, -0.004093046300113201, 0.023584339767694473, 0.004855738487094641, 0.001869852072559297, 0.015508021228015423, -0.00013341776502784342, 0.0027002738788723946, 0.023978549987077713, 0.012103665620088577, -0.001799692981876433, 0.025289611890912056, 0.0016626502620056272, 0.001391877536661923], [0.004761423449963331, -0.010315248742699623, -0.00831927265971899, 0.013614123687148094, -0.0010181502439081669, -0.027362119406461716, -0.00894473772495985, -0.027201855555176735, -0.011980108916759491, -0.008489024825394154, -0.030786853283643723, 0.002281383378431201, 0.013763252645730972, -0.005122210364788771, 0.010090508498251438, 0.0006206255056895316, -0.007705871947109699, -0.0042984141036868095, 0.01830514706671238, 0.013436906039714813, -0.0012944983318448067, -0.006975371856242418, -0.008286849595606327, -0.0133218290284276], [0.0039486452005803585, 0.009505740366876125, 0.0059835538268089294, 0.020908121019601822, 0.031017368659377098, 0.011289125308394432, 0.02710958942770958, 0.014865073375403881, 0.0040222108364105225, -0.0033935094252228737, 0.04924509674310684, -0.006874582264572382, 0.03988483175635338, 0.008642149157822132, -0.0023423575330525637, 0.0362037755548954, 0.005034719593822956, 0.002409904496744275, 0.04643581062555313, 0.01939796283841133, -0.005273358430713415, 0.045358363538980484, 0.0012102776672691107, 0.0004308476345613599], [0.0007311324588954449, 0.0003417038533370942, 0.0010815580608323216, 0.007098470814526081, 0.005151078104972839, -0.003261781530454755, 0.003356963163241744, -0.00046829908387735486, -0.00046832984662614763, -0.0025127315893769264, 0.007724666502326727, -0.0008293692953884602, 0.00999168585985899, 0.0011449099984019995, -0.0009327774751000106, 0.012667280621826649, 0.0011854710755869746, -0.0024394437205046415, 0.01593964919447899, 0.005759133026003838, -0.0029230068903416395, 0.008548175916075706, -0.0008956789970397949, -0.0024843064602464437], [-0.003790806280449033, 0.007245482876896858, 0.005368613172322512, -0.011132129468023777, 0.000928726454731077, 0.019041281193494797, 0.003107326105237007, 0.018655462190508842, 0.008196434006094933, 0.0063867331482470036, 0.018937617540359497, -0.0014257425209507346, -0.010707741603255272, 0.0024529965594410896, -0.005713329184800386, -0.003459569765254855, 0.005257850978523493, 0.004305528476834297, -0.016740204766392708, -0.010524595156311989, 0.0019521620124578476, 0.001106537994928658, 0.005848497152328491, 0.009584013372659683], [-0.008875250816345215, -0.010566015727818012, -0.005940607748925686, -0.04238897189497948, -0.053396448493003845, -0.006100939586758614, -0.0402982123196125, -0.011975713074207306, -0.0012039903085678816, 0.01006928738206625, -0.06812123209238052, 0.010941355489194393, -0.07651636749505997, -0.011973581276834011, -0.00297189736738801, -0.06154012680053711, -0.0036387513391673565, -0.002677848795428872, -0.0879189521074295, -0.04017307236790657, 0.009338394738733768, -0.07332731038331985, 0.0004611083713825792, 0.004399403464049101]],dtype=torch.float32),), ('trf_blocks.1.ff.layers.2.bias', torch.tensor([0.03154725208878517, -0.01670411229133606, 0.06043548882007599, 0.011096147820353508, 0.00923063326627016, -0.09560538828372955],dtype=torch.float32),), ('trf_blocks.1.norm1.scale', torch.tensor([-0.013441844843327999, -0.011842355132102966, 0.008309982717037201, -0.003617989830672741, -0.0019312625518068671, 0.0006413089577108622],dtype=torch.float32),), ('trf_blocks.1.norm1.shift', torch.tensor([-0.019071368500590324, 0.00523535069078207, 0.004887863527983427, 0.00878673791885376, 0.019840974360704422, -0.011967511847615242],dtype=torch.float32),), ('trf_blocks.1.norm2.scale', torch.tensor([0.00013801519526168704, -0.0010525458492338657, -0.017790306359529495, -0.0031325812451541424, -0.0017784049268811941, -0.0053206635639071465],dtype=torch.float32),), ('trf_blocks.1.norm2.shift', torch.tensor([0.0008254545973613858, -0.0025766235776245594, -0.01386751513928175, 0.010208092629909515, 0.006519261747598648, -0.007436468731611967],dtype=torch.float32),), ('final_norm.scale', torch.tensor([0.12889038026332855, 0.1936856061220169, 0.318996787071228, 0.15841543674468994, 0.015579728409647942, -0.00835171714425087],dtype=torch.float32),), ('final_norm.shift', torch.tensor([0.014023208990693092, -0.17722737789154053, 0.2237599641084671, -0.13047362864017487, -0.06825707852840424, -0.15719911456108093],dtype=torch.float32),), ('out_head.weight', torch.tensor([[0.0020471587777137756, -0.03730011358857155, 0.07248109579086304, -0.010023682378232479, -0.010459953919053078, -0.01674451306462288], [-0.047190163284540176, -0.036366987973451614, 0.14438635110855103, -0.013879861682653427, -0.03536875173449516, -0.011580605991184711], [-0.11449801176786423, 0.1317659616470337, -0.059421371668577194, 0.12516188621520996, -0.008598873391747475, -0.0744096115231514], [-0.042400673031806946, 0.0029293405823409557, 0.037099145352840424, 0.02564612403512001, -0.023217465728521347, -5.6480057537555695e-05], [-0.05037529766559601, 0.10500143468379974, -0.09333880245685577, 0.04710974171757698, 0.015000488609075546, -0.023397574201226234], [-0.08228132873773575, 0.1672307848930359, -0.15928755700588226, 0.11865058541297913, 0.021754683926701546, -0.066067174077034], [-0.010923605412244797, 0.08649241179227829, -0.1649051010608673, 0.09541434049606323, 0.022497892379760742, -0.02857593074440956], [0.07114367187023163, -0.009285524487495422, -0.14543454349040985, 0.06189672648906708, 0.025463633239269257, -0.0037839701399207115], [0.008311063051223755, 0.05515258014202118, -0.10377644002437592, -0.019288254901766777, 0.023675989359617233, 0.03592507913708687], [0.03988725692033768, -0.15065908432006836, 0.19979725778102875, -0.14431165158748627, -0.037210021167993546, 0.09249622374773026]],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2110+case)
        cfg={'emb_dim':4+case*2,'vocab_size':9+case,'context_length':6,'n_heads':2,'drop_rate':0.,'qkv_bias':True,'n_layers':2}
        actual=solve_11(cfg)
        observed=observe(actual,'gpt',case,locals())
        same(observed,expected[case])
    print('PASS · 11 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_11()
except Exception as error:
    print('FAIL · 11 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

ID(B,T)→토큰임베딩(B,T,D)+위치(T,D)→블록→최종LN→logits(B,T,V). V는 단어장,context_length는 최대 위치 수입니다.

```python
def solve_11(cfg):
    class GPTModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
            self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
            self.drop_emb = nn.Dropout(cfg["drop_rate"])
            self.trf_blocks = nn.Sequential(
                *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
            self.final_norm = LayerNorm(cfg["emb_dim"])
            self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        def forward(self, in_idx):
            batch_size, seq_len = in_idx.shape
            tok_embeds = self.tok_emb(in_idx)
            pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
            x = tok_embeds + pos_embeds
            x = self.drop_emb(x)
            x = self.trf_blocks(x)
            x = self.final_norm(x)
            logits = self.out_head(x) # 각 단어에 대한 예측 점수 (Logits)
            return logits
    return GPTModel(cfg)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 12. Greedy 생성 루프

원본: `Chapter_4_Excercise_GPT.ipynb` **전체 셀 5** (Markdown 포함, 1부터), 원본 줄 2~31.

**목표:** 원본 빈칸을 완성하세요. 반환값: `generate_text_simple(model, idx, max_new_tokens, context_size)`.

**제공 입력/인자 예시:**
```python
case=0
model=NextTokenModel()
idx=torch.tensor([[1,2,3],[2,3,4]])
max_new_tokens=case+2
context_size=case+2
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_12(model, idx, max_new_tokens, context_size):
    def generate_text_simple(model, idx, max_new_tokens, context_size):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_size:]
            with torch.MISSING():  # ???? 원본 빈칸
                logits = model(idx_cond)
            logits = logits[:, MISSING, :]  # ???? 원본 빈칸
            idx_next = torch.MISSING(logits, dim=-1, keepdim=True)   # ???? 원본 빈칸
            idx = torch.MISSING((idx, idx_next), dim=1)   # ???? 원본 빈칸
        return idx
    return generate_text_simple(model, idx, max_new_tokens, context_size)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_12():
    expected=(torch.tensor([[1, 2, 3, 4, 5], [2, 3, 4, 5, 6]],dtype=torch.int64), torch.tensor([[1, 2, 3, 4, 5, 6], [2, 3, 4, 5, 6, 0]],dtype=torch.int64),)
    for case in range(2):
        torch.manual_seed(2120+case)
        model=NextTokenModel()
        idx=torch.tensor([[1,2,3],[2,3,4]])
        max_new_tokens=case+2
        context_size=case+2
        actual=solve_12(model, idx, max_new_tokens, context_size)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
        assert all(not grad and ids.shape[1]<=context_size for ids,grad in model.calls)
    print('PASS · 12 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_12()
except Exception as error:
    print('FAIL · 12 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

마지막 위치 logits에서 argmax(keepdim=True)로(B,1)을 얻고 시퀀스축 cat. 예: [1,2,3]→[1,2,3,4]. no_grad와 context 자르기도 검사합니다.

```python
def solve_12(model, idx, max_new_tokens, context_size):
    def generate_text_simple(model, idx, max_new_tokens, context_size):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_size:]
            with torch.no_grad():
                logits = model(idx_cond)
            logits = logits[:, -1, :]
            idx_next = torch.argmax(logits, dim=-1, keepdim=True) 
            idx = torch.cat((idx, idx_next), dim=1) 
        return idx
    return generate_text_simple(model, idx, max_new_tokens, context_size)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 13. 사전학습 token loss

원본: `Chapter_5_Excercise_Pretraining.ipynb` **전체 셀 2** (Markdown 포함, 1부터), 원본 줄 2~29.

**목표:** 원본 빈칸을 완성하세요. 반환값: `calc_loss_batch(input_batch,target_batch,model,device)`.

**제공 입력/인자 예시:**
```python
case=0
model=TinyLM()
input_batch=torch.tensor([[1,2,3],[2,4,5]])[:case+1]
target_batch=(input_batch+1)%11
device='cpu'
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_13(input_batch, target_batch, model, device):
    def calc_loss_batch(input_batch, target_batch, model, device):
        input_batch, target_batch = input_batch.to(device), target_batch.to(device)
        logits = model(input_batch)
        loss = torch.nn.functional.MISSING(  # ???? 원본 빈칸
            logits.MISSING(0, 1),   # ???? 원본 빈칸
            target_batch.MISSING()  # ???? 원본 빈칸
        )
        return loss
    return calc_loss_batch(input_batch,target_batch,model,device)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_13():
    expected=((torch.tensor(2.6114633083343506,dtype=torch.float32), (torch.tensor([[0.0, 0.0, 0.0, 0.0], [0.022877514362335205, -0.1405232548713684, -0.15600939095020294, -0.07650303095579147], [-0.011078125797212124, 0.0903296247124672, -0.1233186349272728, -0.0892900601029396], [0.14445483684539795, -0.09439736604690552, 0.126342311501503, 0.03862908482551575], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]],dtype=torch.float32), torch.tensor([[-0.02555627003312111, -0.018296830356121063, 0.12166879326105118, 0.024328824132680893], [-0.019751382991671562, -0.03980554640293121, 0.03521566092967987, 0.002507117111235857], [0.050534121692180634, 0.34072914719581604, -0.011642642319202423, -0.05570029467344284], [0.2751350402832031, 0.3627879023551941, -0.11539433896541595, 0.19235368072986603], [-0.06008383631706238, -0.27562057971954346, -0.4373217225074768, -0.17203299701213837], [-0.010042006149888039, -0.012499505653977394, 0.03758789598941803, 0.007023234851658344], [-0.07882708311080933, -0.12982429563999176, 0.10458964854478836, -0.01547783799469471], [-0.03456206992268562, -0.06876327097415924, 0.0674503892660141, 0.0065752314403653145], [-0.042561210691928864, -0.07955069094896317, 0.04951689392328262, -0.00805571023374796], [-0.034277066588401794, -0.06601651757955551, 0.04288055747747421, -0.0045249564573168755], [-0.02000819891691208, -0.013139806687831879, 0.1054488942027092, 0.023003770038485527]],dtype=torch.float32), torch.tensor([0.12249866873025894, 0.05438975617289543, -0.2593126595020294, -0.1987154632806778, -0.2779994010925293, 0.04185688868165016, 0.15443742275238037, 0.10056993365287781, 0.08407755196094513, 0.07253801077604294, 0.10565927624702454],dtype=torch.float32),),), (torch.tensor(2.324022054672241,dtype=torch.float32), (torch.tensor([[0.0, 0.0, 0.0, 0.0], [-0.051338426768779755, -0.07616361975669861, 0.0400782935321331, -0.022427422925829887], [0.15559741854667664, 0.09774962067604065, 0.011943994089961052, -0.11817272007465363], [-0.01855979673564434, 0.0017660134471952915, -0.08770866692066193, 0.07972985506057739], [-0.025372400879859924, 0.04147248715162277, -0.008592061698436737, 0.026473920792341232], [-0.05662776529788971, -0.022925859317183495, 0.08804095536470413, 0.05162901058793068], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]],dtype=torch.float32), torch.tensor([[0.03190313279628754, -0.057698652148246765, 0.0036886902526021004, 0.005781650077551603], [-0.0089187640696764, -0.03220992162823677, 0.02982402592897415, -0.03589849919080734], [-0.010690051130950451, -0.09047948569059372, 0.010063253343105316, 0.1375667005777359], [-0.10985998809337616, 0.24659383296966553, 0.09841763228178024, -0.11745612323284149], [-0.02212696149945259, -0.015550062991678715, 0.16861580312252045, -0.17784786224365234], [-0.13091664016246796, 0.09259560704231262, -0.22991980612277985, 0.06952445209026337], [0.19544650614261627, 0.04380948096513748, -0.13177482783794403, 0.15570703148841858], [0.04602391645312309, -0.08747982233762741, 0.024230746552348137, -0.00725931953638792], [0.000799904577434063, -0.02026672475039959, 0.0037625152617692947, -0.0080085638910532], [-0.007835391908884048, -0.038673367351293564, 0.034568823873996735, -0.030972378328442574], [0.0161743201315403, -0.04064086824655533, -0.011476829648017883, 0.00886284839361906]],dtype=torch.float32), torch.tensor([0.10025160759687424, 0.07503774762153625, -0.06370312720537186, -0.24693772196769714, -0.04406516253948212, -0.06330757588148117, -0.10600900650024414, 0.14644773304462433, 0.04563282057642937, 0.0719476193189621, 0.08470509946346283],dtype=torch.float32),),),)
    for case in range(2):
        torch.manual_seed(2130+case)
        model=TinyLM()
        input_batch=torch.tensor([[1,2,3],[2,4,5]])[:case+1]
        target_batch=(input_batch+1)%11
        device='cpu'
        actual=solve_13(input_batch, target_batch, model, device)
        observed=observe(actual,'loss',case,locals())
        same(observed,expected[case])
    print('PASS · 13 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_13()
except Exception as error:
    print('FAIL · 13 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

logits(B,T,V)→(B*T,V),정답(B,T)→(B*T,). 모든 토큰을 CE에 전달합니다. 예: B2,T3이면 정답6개.

```python
def solve_13(input_batch, target_batch, model, device):
    def calc_loss_batch(input_batch, target_batch, model, device):
        input_batch, target_batch = input_batch.to(device), target_batch.to(device)
        logits = model(input_batch)
        loss = torch.nn.functional.cross_entropy(
            logits.flatten(0, 1), 
            target_batch.flatten()
        )
        return loss
    return calc_loss_batch(input_batch,target_batch,model,device)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 14. 학습 모드·gradient·업데이트

원본: `Chapter_5_Excercise_Pretraining.ipynb` **전체 셀 3** (Markdown 포함, 1부터), 원본 줄 2~67.

**목표:** 원본 빈칸을 완성하세요. 반환값: `(model.state_dict(), model.training)`.

**제공 입력/인자 예시:**
```python
case=0
model=TinyLM()
model.eval()
optimizer=torch.optim.SGD(model.parameters(),lr=.03+case*.02)
x=torch.tensor([[1,2,3],[3,2,1]])
batches=[(x,(x+1)%11),(x.flip(1),(x.flip(1)+2)%11)]
device='cpu'
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_14(model, optimizer, batches, device):
    model.MISSING()  # ???? 원본 빈칸
    for input_batch, target_batch in batches:
        optimizer.MISSING()  # ???? 원본 빈칸
        loss = MISSING(input_batch, target_batch, model, device)  # ???? 원본 빈칸
        loss.MISSING()  # ???? 원본 빈칸
        optimizer.MISSING()  # ???? 원본 빈칸
    return (model.state_dict(), model.training)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_14():
    expected=(({'emb.weight':torch.tensor([[0.2531782388687134, -2.4572198390960693, 0.04921353980898857, -1.066631555557251], [-1.9629640579223633, -0.47029003500938416, 1.4360953569412231, -0.45054182410240173], [-0.9653550982475281, -1.02500319480896, 0.1571933925151825, 0.2511318624019623], [-0.6774814128875732, -1.3529053926467896, 0.861018180847168, 1.7000224590301514], [0.9807024002075195, -1.034956932067871, -1.4285423755645752, 0.7387653589248657], [1.5347120761871338, -0.9445746541023254, -1.0851640701293945, -0.0520792119204998], [0.19953656196594238, 0.8863668441772461, 2.106649875640869, 0.11603596061468124], [-0.9337418079376221, 0.5408884286880493, 0.35315683484077454, -0.6197343468666077], [0.9622199535369873, 0.35088908672332764, 0.4702835977077484, 1.8701903820037842], [-0.4398307800292969, -0.5118553042411804, 1.4676251411437988, 0.3348921239376068], [0.34243088960647583, -0.42643800377845764, -0.1728489249944687, -1.1814433336257935]],dtype=torch.float32), 'head.weight':torch.tensor([[0.18937885761260986, 0.0812104195356369, 0.07955891638994217, -0.2007063925266266], [-0.41275539994239807, 0.3949410319328308, 0.2781999409198761, -0.4145042598247528], [-0.24413828551769257, 0.04587148502469063, -0.26507702469825745, 0.32040271162986755], [-0.4759620130062103, 0.38860878348350525, 0.16627156734466553, 0.3657357394695282], [0.0809926837682724, -0.08975999057292938, -0.18627575039863586, -0.2676781415939331], [0.3165949583053589, 0.4336777329444885, 0.4306446313858032, -0.3156495690345764], [-0.08270144462585449, -0.42528578639030457, -0.01807415299117565, -0.15895994007587433], [0.3298153579235077, -0.3710898160934448, 0.1963871419429779, -0.2632284164428711], [0.12503811717033386, -0.008413208648562431, -0.4323216378688812, 0.36466479301452637], [0.4404151439666748, -0.304180771112442, 0.2145267128944397, 0.29653212428092957], [0.28571680188179016, 0.3949296176433563, -0.4235095679759979, -0.1656215637922287]],dtype=torch.float32), 'head.bias':torch.tensor([0.46825915575027466, 0.044152602553367615, 0.2960157096385956, -0.47479209303855896, 0.11311382800340652, 0.15565407276153564, -0.4009826183319092, 0.12184906750917435, 0.08683915436267853, -0.2660936117172241, 0.05923990160226822],dtype=torch.float32)}, True,), ({'emb.weight':torch.tensor([[-0.2339085340499878, -0.11143971234560013, 0.1420205980539322, 0.2663608193397522], [0.8560670614242554, -0.5348289012908936, -0.7253512144088745, 0.10593528300523758], [0.28827714920043945, -3.0419843196868896, 1.1746164560317993, 0.7476558089256287], [0.32933247089385986, -0.6326086521148682, -1.4580628871917725, -0.39750680327415466], [1.373531699180603, 1.1894501447677612, -0.5937201976776123, 0.5231421589851379], [2.0878143310546875, 0.3106793761253357, 1.4459551572799683, 0.8130881786346436], [0.442950576543808, 0.650797426700592, -1.0049008131027222, 0.4396786689758301], [0.6427740454673767, 0.10652431100606918, -0.876518726348877, 0.8200178146362305], [0.7615563869476318, 1.5453170537948608, -0.42910611629486084, 0.09006009250879288], [0.009406697936356068, -0.3538094460964203, -0.7220577597618103, 0.030998317524790764], [-0.4353833496570587, -0.07995075732469559, -0.6893588304519653, -0.6202071905136108]],dtype=torch.float32), 'head.weight':torch.tensor([[-0.24361395835876465, 0.06900696456432343, 0.10530220717191696, -0.14207060635089874], [-0.25890859961509705, 0.3260078728199005, 0.43285608291625977, -0.23534052073955536], [-0.4563310146331787, 0.321699857711792, 0.19524091482162476, -0.4972778260707855], [-0.13578231632709503, -0.017731821164488792, 0.24336937069892883, -0.20456378161907196], [0.11112543195486069, 0.03755604848265648, -0.20158477127552032, 0.0342618003487587], [-0.14159995317459106, 0.05884794890880585, -0.38759687542915344, 0.4276222288608551], [-0.33210116624832153, -0.07911334186792374, 0.10593529790639877, 0.4580012559890747], [-0.045661456882953644, -0.14616645872592926, -0.1931498497724533, 0.3234791159629822], [0.11617136746644974, 0.1079997792840004, 0.2774510085582733, -0.4403722584247589], [0.34250813722610474, 0.12059587240219116, -0.15404823422431946, 0.36161693930625916], [0.011347543448209763, 0.03492680564522743, -0.10688840597867966, -0.30929991602897644]],dtype=torch.float32), 'head.bias':torch.tensor([-0.26017266511917114, -0.3213566839694977, -0.15430884063243866, 0.058511193841695786, 0.40331101417541504, 0.25829657912254333, 0.09558470547199249, 0.3065275549888611, -0.03566942363977432, -0.09260199218988419, 0.0615919791162014],dtype=torch.float32)}, True,),)
    for case in range(2):
        torch.manual_seed(2140+case)
        model=TinyLM()
        model.eval()
        optimizer=torch.optim.SGD(model.parameters(),lr=.03+case*.02)
        x=torch.tensor([[1,2,3],[3,2,1]])
        batches=[(x,(x+1)%11),(x.flip(1),(x.flip(1)+2)%11)]
        device='cpu'
        actual=solve_14(model, optimizer, batches, device)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 14 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_14()
except Exception as error:
    print('FAIL · 14 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

train→zero_grad→loss→backward→step. 두 배치를 연속 처리해 gradient 초기화 누락도 확인합니다. 모델·optimizer는 테스트가 준비하며 전체 epoch는 제외합니다.

```python
def solve_14(model, optimizer, batches, device):
    model.train()
    for input_batch, target_batch in batches:
        optimizer.zero_grad()
        loss = calc_loss_batch(input_batch, target_batch, model, device)
        loss.backward()
        optimizer.step()

    return (model.state_dict(), model.training)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 15. EOS가 있는 Greedy 생성

원본: `Chapter_5_Excercise_Pretraining.ipynb` **전체 셀 4** (Markdown 포함, 1부터), 원본 줄 2~52.

**목표:** 원본 빈칸을 완성하세요. 반환값: `generate(model,idx,max_new_tokens,context_size,eos_id)`.

**제공 입력/인자 예시:**
```python
case=0
model=NextTokenModel()
idx=torch.tensor([[1,2]])
max_new_tokens=5
context_size=3
eos_id=[None,4][case]
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_15(model, idx, max_new_tokens, context_size, eos_id):
    def generate(model, idx, max_new_tokens, context_size, eos_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_size:]
            with torch.no_grad():
                logits = model(idx_cond)
            logits = logits[:, MISSING, :]  # ???? 원본 빈칸
            idx_next = torch.MISSING(logits, dim=-1, keepdim=True)  # ???? 원본 빈칸
            if idx_next == eos_id:
                break
            idx = torch.cat((idx, MISSING), dim=1)  # ???? 원본 빈칸
        return idx
    return generate(model,idx,max_new_tokens,context_size,eos_id)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_15():
    expected=(torch.tensor([[1, 2, 3, 4, 5, 6, 0]],dtype=torch.int64), torch.tensor([[1, 2, 3]],dtype=torch.int64),)
    for case in range(2):
        torch.manual_seed(2150+case)
        model=NextTokenModel()
        idx=torch.tensor([[1,2]])
        max_new_tokens=5
        context_size=3
        eos_id=[None,4][case]
        actual=solve_15(model, idx, max_new_tokens, context_size, eos_id)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 15 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_15()
except Exception as error:
    print('FAIL · 15 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

원본은 주석과 달리 실제로 greedy만 구현합니다. 마지막 토큰을 뽑고 EOS이면 붙이기 전에 종료합니다. B=1로 제한: 원본 if문은 다중 배치에 맞지 않습니다.

```python
def solve_15(model, idx, max_new_tokens, context_size, eos_id):
    def generate(model, idx, max_new_tokens, context_size, eos_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_size:]
            with torch.no_grad():
                logits = model(idx_cond)
            logits = logits[:, -1, :]
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
            if idx_next == eos_id:
                break
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    return generate(model,idx,max_new_tokens,context_size,eos_id)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 16. 텍스트에 배치 차원 추가

원본: `Chapter_5_Excercise_Pretraining.ipynb` **전체 셀 5** (Markdown 포함, 1부터), 원본 줄 1~14.

**목표:** 원본 빈칸을 완성하세요. 반환값: `text_to_token_ids(text,tokenizer)`.

**제공 입력/인자 예시:**
```python
case=0
text=['abc','Hello'][case]
tokenizer=DummyTokenizer()
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_16(text, tokenizer):
    def text_to_token_ids(text, tokenizer):
        encoded = tokenizer.encode(text)
        encoded_tensor = torch.tensor(encoded).MISSING(0)   # ???? 원본 빈칸
        return encoded_tensor
    return text_to_token_ids(text,tokenizer)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_16():
    expected=(torch.tensor([[97, 98, 99]],dtype=torch.int64), torch.tensor([[72, 101, 108, 108, 111]],dtype=torch.int64),)
    for case in range(2):
        torch.manual_seed(2160+case)
        text=['abc','Hello'][case]
        tokenizer=DummyTokenizer()
        actual=solve_16(text, tokenizer)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 16 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_16()
except Exception as error:
    print('FAIL · 16 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

encode 결과 길이T → tensor(T,) → unsqueeze(0)=(1,T). 예: [65,66] → [[65,66]].

```python
def solve_16(text, tokenizer):
    def text_to_token_ids(text, tokenizer):
        encoded = tokenizer.encode(text)
        encoded_tensor = torch.tensor(encoded).unsqueeze(0) 
        return encoded_tensor
    return text_to_token_ids(text,tokenizer)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 17. 토큰에서 배치 차원 제거

원본: `Chapter_5_Excercise_Pretraining.ipynb` **전체 셀 5** (Markdown 포함, 1부터), 원본 줄 17~26.

**목표:** 원본 빈칸을 완성하세요. 반환값: `token_ids_to_text(token_ids,tokenizer)`.

**제공 입력/인자 예시:**
```python
case=0
tokenizer=DummyTokenizer()
token_ids=torch.tensor([tokenizer.encode(['abc','Hello'][case])])
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_17(token_ids, tokenizer):
    def token_ids_to_text(token_ids, tokenizer):
        flat = token_ids.MISSING(0)   # ???? 원본 빈칸
        return tokenizer.decode(flat.tolist())
    return token_ids_to_text(token_ids,tokenizer)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_17():
    expected=('abc', 'Hello',)
    for case in range(2):
        torch.manual_seed(2170+case)
        tokenizer=DummyTokenizer()
        token_ids=torch.tensor([tokenizer.encode(['abc','Hello'][case])])
        actual=solve_17(token_ids, tokenizer)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 17 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_17()
except Exception as error:
    print('FAIL · 17 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

(1,T)→squeeze(0)→(T,)→tolist→decode. B=1이라는 전제입니다.

```python
def solve_17(token_ids, tokenizer):
    def token_ids_to_text(token_ids, tokenizer):
        flat = token_ids.squeeze(0) 
        return tokenizer.decode(flat.tolist())
    return token_ids_to_text(token_ids,tokenizer)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 18. AdamW 파라미터와 학습률

원본: `Chapter_5_Excercise_Pretraining.ipynb` **전체 셀 5** (Markdown 포함, 1부터), 원본 줄 142~144.

**목표:** 원본 빈칸을 완성하세요. 반환값: `optimizer`.

**제공 입력/인자 예시:**
```python
case=0
model=TinyLM()
settings={'learning_rate':.002*(case+1),'weight_decay':.1+case*.1}
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_18(model, settings):
    optimizer = torch.optim.AdamW(
        model.MISSING(), MISSING=settings["learning_rate"], weight_decay=settings["weight_decay"]  # ???? 원본 빈칸
    )
    return optimizer

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_18():
    expected=((0.002, 0.1,), (0.004, 0.2,),)
    for case in range(2):
        torch.manual_seed(2180+case)
        model=TinyLM()
        settings={'learning_rate':.002*(case+1),'weight_decay':.1+case*.1}
        actual=solve_18(model, settings)
        observed=observe(actual,'optimizer',case,locals())
        same(observed,expected[case])
    print('PASS · 18 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_18()
except Exception as error:
    print('FAIL · 18 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

AdamW(model.parameters(),lr=...,weight_decay=...). 모든 학습 파라미터가 전달됐는지와 설정을 검사합니다.

```python
def solve_18(model, settings):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"]
    )
    return optimizer
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 19. 분류 데이터 자르기와 패딩

원본: `Chapter_6_Excercise_Finetuning_Classification.ipynb` **전체 셀 1** (Markdown 포함, 1부터), 원본 줄 109~121.

**목표:** 원본 빈칸을 완성하세요. 반환값: `self.encoded_texts`.

**제공 입력/인자 예시:**
```python
case=0
self=SimpleNamespace(max_length=case+3,encoded_texts=[[1,2],[3,4,5,6,7],[]])
pad_token_id=8+case
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_19(self, pad_token_id):
    self.encoded_texts = [
    encoded_text[:MISSING]  # ???? 원본 빈칸
    for encoded_text in self.encoded_texts
    ]
    self.encoded_texts = [
    encoded_text + [MISSING] * (self.max_length - len(encoded_text))  # ???? 원본 빈칸
    for encoded_text in self.encoded_texts
    ]
    return self.encoded_texts

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_19():
    expected=(((1, 2, 8,), (3, 4, 5,), (8, 8, 8,),), ((1, 2, 9, 9,), (3, 4, 5, 6,), (9, 9, 9, 9,),),)
    for case in range(2):
        torch.manual_seed(2190+case)
        self=SimpleNamespace(max_length=case+3,encoded_texts=[[1,2],[3,4,5,6,7],[]])
        pad_token_id=8+case
        actual=solve_19(self, pad_token_id)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 19 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_19()
except Exception as error:
    print('FAIL · 19 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

최대길이로 먼저 자르고 부족한 만큼 pad를 붙입니다. 예: max3,pad9이면 [1]→[1,9,9], [1,2,3,4]→[1,2,3]. CSV 다운로드는 제외했습니다.

```python
def solve_19(self, pad_token_id):
    self.encoded_texts = [
    encoded_text[:self.max_length]
    for encoded_text in self.encoded_texts
    ]
    self.encoded_texts = [
    encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
    for encoded_text in self.encoded_texts
    ]
    return self.encoded_texts
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 20. 분류 정답 열 선택

원본: `Chapter_6_Excercise_Finetuning_Classification.ipynb` **전체 셀 1** (Markdown 포함, 1부터), 원본 줄 127~127.

**목표:** 원본 빈칸을 완성하세요. 반환값: `label`.

**제공 입력/인자 예시:**
```python
case=0
self=SimpleNamespace(data=MiniFrame([{'Label':0,'Other':8},{'Label':1,'Other':9}]))
index=case
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_20(self, index):
    label = self.data.iloc[index]["MISSING"]  # ???? 원본 빈칸
    return label

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_20():
    expected=(0, 1,)
    for case in range(2):
        torch.manual_seed(2200+case)
        self=SimpleNamespace(data=MiniFrame([{'Label':0,'Other':8},{'Label':1,'Other':9}]))
        index=case
        actual=solve_20(self, index)
        observed=observe(actual,'value',case,locals())
        same(observed,expected[case])
    print('PASS · 20 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_20()
except Exception as error:
    print('FAIL · 20 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

iloc[index]로 해당 행을 찾고 Label 열에서 정답을 얻습니다. 예: index1→Label1. pandas 설치 없이 동일 인터페이스 더미를 제공합니다.

```python
def solve_20(self, index):
    label = self.data.iloc[index]["Label"]
    return label
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 21. 분류는 마지막 위치 logits

원본: `Chapter_6_Excercise_Finetuning_Classification.ipynb` **전체 셀 1** (Markdown 포함, 1부터), 원본 줄 180~190.

**목표:** 원본 빈칸을 완성하세요. 반환값: `calc_loss_batch(input_batch,target_batch,model,device)`.

**제공 입력/인자 예시:**
```python
case=0
model=TinyLM(vocab=3)
input_batch=torch.tensor([[0,1,2],[1,2,0]])[:case+1]
target_batch=torch.tensor([1,0])[:case+1]
device='cpu'
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_21(input_batch, target_batch, model, device):
    def calc_loss_batch(input_batch, target_batch, model, device):
        input_batch, target_batch = input_batch.to(device), target_batch.to(device)
        logits = model(input_batch)[:, MISSING, :]    # ???? 원본 빈칸
        loss = torch.nn.functional.cross_entropy(logits, target_batch)
        return loss
    return calc_loss_batch(input_batch,target_batch,model,device)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_21():
    expected=((torch.tensor(1.5486772060394287,dtype=torch.float32), (torch.tensor([[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.258014440536499, -0.45066362619400024, -0.15976756811141968, -0.019019827246665955]],dtype=torch.float32), torch.tensor([[0.30000802874565125, -0.3693217635154724, 0.07930586487054825, 0.16840963065624237], [-0.7932078838348389, 0.9764702916145325, -0.2096811681985855, -0.44526758790016174], [0.49319982528686523, -0.6071485280990601, 0.13037529587745667, 0.2768579423427582]],dtype=torch.float32), torch.tensor([0.2978382408618927, -0.7874710559844971, 0.489632785320282],dtype=torch.float32),),), (torch.tensor(1.2435848712921143,dtype=torch.float32), (torch.tensor([[-0.050671931356191635, 0.08753731846809387, -0.20423400402069092, -0.20464734733104706], [0.0, 0.0, 0.0, 0.0], [-0.1120954379439354, -0.05481188744306564, -0.010575242340564728, -0.009106345474720001]],dtype=torch.float32), torch.tensor([[0.10044623166322708, 0.1490955352783203, 0.24372044205665588, -0.010828904807567596], [-0.5474966764450073, 0.340444415807724, 0.26557958126068115, 0.1957564651966095], [0.4470504820346832, -0.4895400106906891, -0.5093001127243042, -0.1849275678396225]],dtype=torch.float32), torch.tensor([-0.32027044892311096, -0.13103589415550232, 0.45130640268325806],dtype=torch.float32),),),)
    for case in range(2):
        torch.manual_seed(2210+case)
        model=TinyLM(vocab=3)
        input_batch=torch.tensor([[0,1,2],[1,2,0]])[:case+1]
        target_batch=torch.tensor([1,0])[:case+1]
        device='cpu'
        actual=solve_21(input_batch, target_batch, model, device)
        observed=observe(actual,'loss',case,locals())
        same(observed,expected[case])
    print('PASS · 21 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_21()
except Exception as error:
    print('FAIL · 21 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

분류 logits(B,T,C)에서 [:,-1,:]→(B,C). 정답은(B,). 사전학습처럼 모든 토큰을 flatten하지 않습니다.

```python
def solve_21(input_batch, target_batch, model, device):
    def calc_loss_batch(input_batch, target_batch, model, device):
        input_batch, target_batch = input_batch.to(device), target_batch.to(device)
        logits = model(input_batch)[:, -1, :]  
        loss = torch.nn.functional.cross_entropy(logits, target_batch)
        return loss
    return calc_loss_batch(input_batch,target_batch,model,device)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 22. 분류 미세조정: 동결·출력층·일부 해제

원본: `Chapter_6_Excercise_Finetuning_Classification.ipynb` **전체 셀 2** (Markdown 포함, 1부터), 원본 줄 144~171.

**목표:** 원본 빈칸을 완성하세요. 반환값: `(model, flags(model))`.

**제공 입력/인자 예시:**
```python
case=0
model=TinyClassifier()
BASE_CONFIG={'emb_dim':4}
num_classes=case+2
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_22(model, BASE_CONFIG, num_classes):
    for param in model.parameters():
        param.requires_grad = MISSING  # ???? 원본 빈칸


    model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=MISSING)  # ???? 원본 빈칸

    for param in model.trf_blocks[MISSING].parameters():  # ???? 원본 빈칸
        param.requires_grad = MISSING  # ???? 원본 빈칸
    for param in model.final_norm.parameters():
        param.requires_grad = MISSING  # ???? 원본 빈칸
    return (model, flags(model))

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_22():
    expected=((torch.tensor([[[0.5718974471092224, -0.734318733215332], [0.18803328275680542, -0.1738000512123108]], [[0.54616379737854, -0.9042273163795471], [0.5414677858352661, -0.9321832060813904]]],dtype=torch.float32), {'emb.weight':False, 'trf_blocks.0.weight':False, 'trf_blocks.0.bias':False, 'trf_blocks.1.weight':True, 'trf_blocks.1.bias':True, 'final_norm.weight':True, 'final_norm.bias':True, 'out_head.weight':True, 'out_head.bias':True}, (2, 4,),), (torch.tensor([[[-0.7648789882659912, -0.4108617901802063, -0.057131052017211914], [-0.5638635754585266, -0.5504590272903442, 0.1808064877986908]], [[-0.7039250135421753, -0.2600558400154114, -0.48667311668395996], [-0.31123965978622437, -0.5838701725006104, 0.533568263053894]]],dtype=torch.float32), {'emb.weight':False, 'trf_blocks.0.weight':False, 'trf_blocks.0.bias':False, 'trf_blocks.1.weight':True, 'trf_blocks.1.bias':True, 'final_norm.weight':True, 'final_norm.bias':True, 'out_head.weight':True, 'out_head.bias':True}, (3, 4,),),)
    for case in range(2):
        torch.manual_seed(2220+case)
        model=TinyClassifier()
        BASE_CONFIG={'emb_dim':4}
        num_classes=case+2
        actual=solve_22(model, BASE_CONFIG, num_classes)
        observed=observe(actual,'classification',case,locals())
        same(observed,expected[case])
    print('PASS · 22 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_22()
except Exception as error:
    print('FAIL · 22 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

전체 동결 후 새 분류 head를 만들고 마지막 Transformer와 final_norm만 해제합니다. 새 head는 기본 requires_grad=True. 예: 이전 블록False,마지막블록True.

```python
def solve_22(model, BASE_CONFIG, num_classes):
    for param in model.parameters():
        param.requires_grad = False


    model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

    for param in model.trf_blocks[-1].parameters():
        param.requires_grad = True
    for param in model.final_norm.parameters():
        param.requires_grad = True
    return (model, flags(model))
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 23. LoRA A·B와 저랭크 곱

원본: `Chapter_6_Excercise_Finetuning_Classification_LoRA.ipynb` **전체 셀 3** (Markdown 포함, 1부터), 원본 줄 5~29.

**목표:** 원본 빈칸을 완성하세요. 반환값: `LoRALayer(in_dim,out_dim,rank,alpha)`.

**제공 입력/인자 예시:**
```python
case=0
in_dim=3+case
out_dim=5
rank=case+1
alpha=3.+case
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_23(in_dim, out_dim, rank, alpha):
    class LoRALayer(nn.Module):
        def __init__(self, in_dim, out_dim, rank, alpha):
            super().__init__()
            self.A = nn.Parameter(torch.empty(in_dim, MISSING))  # ???? 원본 빈칸
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(MISSING, out_dim))  # ???? 원본 빈칸
            self.alpha = alpha
            self.rank = rank
        def forward(self, x):
            x = (self.MISSING / self.rank) * (x @ self.MISSING @ self.MISSING)  # ???? 원본 빈칸
            return x
    return LoRALayer(in_dim,out_dim,rank,alpha)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_23():
    expected=((torch.tensor([[[0.15759655833244324, 0.23639483749866486, 0.3151931166648865, 0.3939913809299469, 0.4727896749973297], [0.10071009397506714, 0.1510651558637619, 0.20142018795013428, 0.25177526473999023, 0.3021303117275238], [0.03589668124914169, 0.05384502187371254, 0.07179336249828339, 0.08974169194698334, 0.10769004374742508]], [[-0.2222379744052887, -0.33335697650909424, -0.4444759488105774, -0.5555949807167053, -0.6667139530181885], [0.15812045335769653, 0.2371806800365448, 0.31624090671539307, 0.39530110359191895, 0.4743613600730896], [0.16596809029579163, 0.24895213544368744, 0.33193618059158325, 0.41492021083831787, 0.4979042708873749]]],dtype=torch.float32), {'A':True, 'B':True}, (('A', torch.tensor([[-0.3400391936302185], [0.029755301773548126], [-0.17682883143424988]],dtype=torch.float32),), ('B', torch.tensor([[0.09213665127754211, 0.13820497691631317, 0.18427330255508423, 0.23034164309501648, 0.27640995383262634]],dtype=torch.float32),),),), (torch.tensor([[[0.1719110608100891, 0.20637144148349762, 0.24083182215690613, 0.27529221773147583, 0.30975258350372314], [0.13019704818725586, 0.1453813761472702, 0.16056567430496216, 0.1757500022649765, 0.19093433022499084], [0.6515415906906128, 0.9597291946411133, 1.2679166793823242, 1.5761042833328247, 1.8842918872833252]], [[-0.33341681957244873, -0.38123586773872375, -0.42905494570732117, -0.4768739938735962, -0.5246930718421936], [0.1994149386882782, 0.16321900486946106, 0.12702307105064392, 0.09082716703414917, 0.05463123321533203], [0.4333886504173279, 0.46156132221221924, 0.489734023809433, 0.5179067254066467, 0.5460793972015381]]],dtype=torch.float32), {'A':True, 'B':True}, (('A', torch.tensor([[0.018518339842557907, 0.022985897958278656], [-0.1377822458744049, -0.27135154604911804], [-0.13951367139816284, -0.2900141477584839], [0.703362226486206, 1.4966657161712646]],dtype=torch.float32),), ('B', torch.tensor([[0.21271055936813354, 0.3396128714084625, 0.4665151536464691, 0.5934175252914429, 0.720319926738739], [0.09346745908260345, 0.10094936192035675, 0.10843126475811005, 0.11591318249702454, 0.12339508533477783]],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2230+case)
        in_dim=3+case
        out_dim=5
        rank=case+1
        alpha=3.+case
        actual=solve_23(in_dim, out_dim, rank, alpha)
        observed=observe(actual,'lora',case,locals())
        same(observed,expected[case])
    print('PASS · 23 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_23()
except Exception as error:
    print('FAIL · 23 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

A=(in,rank),B=(rank,out). Δy=(alpha/rank)*x@A@B. 초기B=0이면 Δy=0. 검증에서는 B를0이 아닌 값으로 바꿔 수식과 A/B gradient도 확인합니다.

```python
def solve_23(in_dim, out_dim, rank, alpha):
    class LoRALayer(nn.Module):
        def __init__(self, in_dim, out_dim, rank, alpha):
            super().__init__()
            self.A = nn.Parameter(torch.empty(in_dim, rank))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(rank, out_dim))
            self.alpha = alpha
            self.rank = rank
        def forward(self, x):
            x = (self.alpha / self.rank) * (x @ self.A @ self.B)
            return x
    return LoRALayer(in_dim,out_dim,rank,alpha)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 24. 기존 Linear에 LoRA 출력 더하기

원본: `Chapter_6_Excercise_Finetuning_Classification_LoRA.ipynb` **전체 셀 3** (Markdown 포함, 1부터), 원본 줄 31~46.

**목표:** 원본 빈칸을 완성하세요. 반환값: `LinearWithLoRA(linear,rank,alpha)`.

**제공 입력/인자 예시:**
```python
case=0
linear=nn.Linear(3,4)
rank=case+1
alpha=3.
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_24(linear, rank, alpha):
    class LinearWithLoRA(nn.Module):
        def __init__(self, linear, rank, alpha):
            super().__init__()
            self.linear = linear
            self.lora = LoRALayer(
                linear.in_features, linear.out_features, rank, alpha
            )
        def forward(self, x):
            return self.linear(x) + self.MISSING(x)  # ???? 원본 빈칸
    return LinearWithLoRA(linear,rank,alpha)

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_24():
    expected=((torch.tensor([[[0.006127595901489258, -0.2708471417427063, 0.6082639694213867, 0.5079018473625183], [0.21542629599571228, -0.4055670201778412, 0.3459470272064209, 0.23709237575531006], [0.6034576296806335, -0.24365851283073425, 0.029225051403045654, 0.4902263283729553]], [[-0.018122833222150803, -0.3739796578884125, 0.4992074966430664, 0.805986762046814], [0.269470751285553, -0.37744325399398804, 0.2752508819103241, 0.43714284896850586], [0.3493555784225464, -0.7725666761398315, -0.03837290406227112, 0.0881035327911377]]],dtype=torch.float32), {'linear.weight':True, 'linear.bias':True, 'lora.A':True, 'lora.B':True}, (('linear.weight', torch.tensor([[0.025502052158117294, 0.017445605248212814, 0.019779466092586517], [0.0801318883895874, 0.053413376212120056, -0.04709315672516823], [-0.07278484106063843, -0.06113610044121742, 0.09914438426494598], [-0.02681133709847927, -0.09355838596820831, 0.10718764364719391]],dtype=torch.float32),), ('linear.bias', torch.tensor([0.11880959570407867, -0.20367185771465302, 0.14329345524311066, 0.21387115120887756],dtype=torch.float32),), ('lora.A', torch.tensor([[-0.020069435238838196], [-0.07758074998855591], [0.12461929023265839]],dtype=torch.float32),), ('lora.B', torch.tensor([[0.0795086994767189, 0.029779523611068726, 0.08604627847671509, 0.17161774635314941]],dtype=torch.float32),),),), (torch.tensor([[[0.6686171293258667, -0.01996089518070221, -0.06068316102027893, 1.038588047027588], [0.8598000407218933, 0.7445059418678284, -1.7708826065063477, -0.10243208706378937], [-0.3023022413253784, -0.001960188150405884, 0.7995785474777222, 1.1016297340393066]], [[0.40466225147247314, 0.9420152902603149, -0.561103343963623, 0.482155978679657], [0.4472850561141968, 0.8137636184692383, 0.4791637361049652, 1.2442547082901], [0.45193853974342346, -0.6073049306869507, 1.0037410259246826, 1.701157569885254]]],dtype=torch.float32), {'linear.weight':True, 'linear.bias':True, 'lora.A':True, 'lora.B':True}, (('linear.weight', torch.tensor([[-0.13589859008789062, 0.14386913180351257, -0.14503341913223267], [-0.027547687292099, 0.270901620388031, 0.019469991326332092], [-0.22409573197364807, -0.5153540968894958, 0.09568920731544495], [-0.48923471570014954, -0.29893165826797485, -0.1234566867351532]],dtype=torch.float32),), ('linear.bias', torch.tensor([0.21083340048789978, 0.1559215784072876, -0.009182155132293701, 0.45544618368148804],dtype=torch.float32),), ('lora.A', torch.tensor([[-0.2772747576236725, -0.5403077602386475], [-0.1841723471879959, -0.30402684211730957], [-0.03496374934911728, -0.08096300810575485]],dtype=torch.float32),), ('lora.B', torch.tensor([[-0.15000849962234497, -0.24609029293060303, 0.8298124074935913, 0.6584713459014893], [0.12292961776256561, 0.0995061993598938, -0.17120645940303802, 0.02645089663565159]],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2240+case)
        linear=nn.Linear(3,4)
        rank=case+1
        alpha=3.
        actual=solve_24(linear, rank, alpha)
        observed=observe(actual,'wrapped',case,locals())
        same(observed,expected[case])
    print('PASS · 24 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_24()
except Exception as error:
    print('FAIL · 24 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

y=linear(x)+lora(x). 두 출력 모두(B,out)이라 더할 수 있습니다. B를0이 아닌 값으로 바꿔 잔차 누락을 검사합니다.

```python
def solve_24(linear, rank, alpha):
    class LinearWithLoRA(nn.Module):
        def __init__(self, linear, rank, alpha):
            super().__init__()
            self.linear = linear
            self.lora = LoRALayer(
                linear.in_features, linear.out_features, rank, alpha
            )
        def forward(self, x):
            return self.linear(x) + self.lora(x)
    return LinearWithLoRA(linear,rank,alpha)
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 25. 중첩 Linear를 재귀 교체

원본: `Chapter_6_Excercise_Finetuning_Classification_LoRA.ipynb` **전체 셀 3** (Markdown 포함, 1부터), 원본 줄 49~61.

**목표:** 원본 빈칸을 완성하세요. 반환값: `model`.

**제공 입력/인자 예시:**
```python
case=0
model=nn.Sequential(nn.Linear(3,4),nn.Sequential(nn.ReLU(),nn.Linear(4,2)))
rank=case+1
alpha=2.
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_25(model, rank, alpha):
    def replace_linear_with_lora(model, rank, alpha):
        for name, module in model.named_children():
            if isinstance(module, torch.nn.MISSING):  # ???? 원본 빈칸
                setattr(model, name, LinearWithLoRA(module, rank, alpha))
            else:
                replace_linear_with_lora(module, rank, alpha)
    replace_linear_with_lora(model,rank,alpha)
    return model

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_25():
    expected=((torch.tensor([[[-0.5330086946487427, 0.08056756854057312], [-0.2595175802707672, 0.09514133632183075], [-0.22959710657596588, 0.20072504878044128]], [[-0.44537749886512756, -0.21540984511375427], [-0.4547327160835266, -0.25544506311416626], [-0.8885506391525269, -1.0153486728668213]]],dtype=torch.float32), {'0.linear.weight':True, '0.linear.bias':True, '0.lora.A':True, '0.lora.B':True, '1.1.linear.weight':True, '1.1.linear.bias':True, '1.1.lora.A':True, '1.1.lora.B':True}, (('0.linear.weight', torch.tensor([[0.008305644616484642, 0.05405803769826889, 0.056920863687992096], [-0.007807753048837185, 0.0048040044493973255, 0.025331061333417892], [0.0563049279153347, -0.2707745432853699, -0.2929091453552246], [0.039229266345500946, -0.21314801275730133, -0.2326306402683258]],dtype=torch.float32),), ('0.linear.bias', torch.tensor([0.0290061105042696, 0.019800368696451187, 0.21595853567123413, 0.1645129919052124],dtype=torch.float32),), ('0.lora.A', torch.tensor([[0.04145541042089462], [-0.20263102650642395], [-0.2144954800605774]],dtype=torch.float32),), ('0.lora.B', torch.tensor([[0.0014004893600940704, 0.005841910373419523, -0.11965088546276093, -0.08781032264232635]],dtype=torch.float32),), ('1.1.linear.weight', torch.tensor([[-0.08741872757673264, -0.0038293113466352224, -0.2954636216163635, -0.250337690114975], [0.013213882222771645, 0.003347771940752864, -0.2679463028907776, -0.2525010108947754]],dtype=torch.float32),), ('1.1.linear.bias', torch.tensor([-0.46846407651901245, -0.18496161699295044],dtype=torch.float32),), ('1.1.lora.A', torch.tensor([[-0.013519582338631153], [0.0002384693070780486], [-0.13947661221027374], [-0.12581783533096313]],dtype=torch.float32),), ('1.1.lora.B', torch.tensor([[1.0756785869598389, 0.8460737466812134]],dtype=torch.float32),),),), (torch.tensor([[[0.2588235139846802, -0.0733538344502449], [0.1930653154850006, -0.2839125096797943], [0.24483126401901245, -0.002032017335295677]], [[0.21203884482383728, 0.05178344249725342], [0.2103678584098816, -0.1060081496834755], [0.1356736421585083, -0.23252174258232117]]],dtype=torch.float32), {'0.linear.weight':True, '0.linear.bias':True, '0.lora.A':True, '0.lora.B':True, '1.1.linear.weight':True, '1.1.linear.bias':True, '1.1.lora.A':True, '1.1.lora.B':True}, (('0.linear.weight', torch.tensor([[0.004248259589076042, -0.027816060930490494, 0.006161958444863558], [-0.019830577075481415, 0.008160529658198357, -0.008680405095219612], [-0.0010120392544195056, -0.005074137821793556, -0.004949596710503101], [-0.020427916198968887, 0.011598825454711914, -0.011760516092181206]],dtype=torch.float32),), ('0.linear.bias', torch.tensor([0.016567299142479897, 0.010152437724173069, -0.006136119831353426, 0.024347111582756042],dtype=torch.float32),), ('0.lora.A', torch.tensor([[-0.007859147153794765, -0.015263602137565613], [0.0003273524343967438, -0.002298816107213497], [-0.0046159131452441216, -0.00846162624657154]],dtype=torch.float32),), ('0.lora.B', torch.tensor([[-0.003796705976128578, -0.005534434225410223, -0.0019476200686767697, -0.0053516714833676815], [-0.0062261708080768585, -0.004496854729950428, -0.0054137795232236385, -0.005305658560246229]],dtype=torch.float32),), ('1.1.linear.weight', torch.tensor([[0.05842704325914383, 0.04762263596057892, 0.040101680904626846, 0.07610568404197693], [-0.0600980781018734, -0.05034759268164635, -0.020207960158586502, -0.028204167261719704]],dtype=torch.float32),), ('1.1.linear.bias', torch.tensor([0.20913341641426086, -0.10767412930727005],dtype=torch.float32),), ('1.1.lora.A', torch.tensor([[-0.0031720083206892014, -0.0033391115721315145], [-0.002789875026792288, -0.0030623702332377434], [0.0009789743926376104, 0.002968346932902932], [0.0033799444790929556, 0.008170097135007381]],dtype=torch.float32),), ('1.1.lora.B', torch.tensor([[-0.028770098462700844, -0.005964287556707859], [0.016708234325051308, -0.014783711172640324]],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2250+case)
        model=nn.Sequential(nn.Linear(3,4),nn.Sequential(nn.ReLU(),nn.Linear(4,2)))
        rank=case+1
        alpha=2.
        actual=solve_25(model, rank, alpha)
        observed=observe(actual,'replace',case,locals())
        same(observed,expected[case])
    print('PASS · 25 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_25()
except Exception as error:
    print('FAIL · 25 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

named_children으로 자식 순회. Linear이면 wrapper로 교체하고, 아니면 안쪽으로 재귀합니다. 중첩 Sequential의 Linear도 교체되어야 합니다.

```python
def solve_25(model, rank, alpha):
    def replace_linear_with_lora(model, rank, alpha):
        for name, module in model.named_children():
            if isinstance(module, torch.nn.Linear):
                setattr(model, name, LinearWithLoRA(module, rank, alpha))
            else:
                replace_linear_with_lora(module, rank, alpha)
    replace_linear_with_lora(model,rank,alpha)
    return model
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>

## 26. 기존 가중치 동결 후 LoRA 주입

원본: `Chapter_6_Excercise_Finetuning_Classification_LoRA.ipynb` **전체 셀 4** (Markdown 포함, 1부터), 원본 줄 64~70.

**목표:** 원본 빈칸을 완성하세요. 반환값: `model`.

**제공 입력/인자 예시:**
```python
case=0
model=nn.Sequential(nn.Linear(3,4),nn.Sequential(nn.ReLU(),nn.Linear(4,2)))
LORA_RANK=case+1
LORA_ALPHA=2.
```

테스트는 서로 다른 입력 두 개를 사용합니다. 작은 CPU 텐서로 shape·값·필요 시 gradient를 검사합니다.

In [ ]:
import torch
import torch.nn as nn

def solve_26(model, LORA_RANK, LORA_ALPHA):
    for param in model.parameters():
        param.requires_grad = MISSING  # ???? 원본 빈칸
    MISSING(model, rank=LORA_RANK, alpha=LORA_ALPHA)  # ???? 원본 빈칸
    return model

In [ ]:
# 정답/실패 확인. 이 셀의 입력과 기대값은 수정하지 마세요.
def check_26():
    expected=((torch.tensor([[[0.5777877569198608, -1.3874906301498413], [0.3195827007293701, -0.35186293721199036], [0.27029383182525635, -0.3203788995742798]], [[0.4575228989124298, -0.6765130758285522], [0.48725858330726624, -0.679935097694397], [0.4185255765914917, -0.9302685260772705]]],dtype=torch.float32), {'0.linear.weight':False, '0.linear.bias':False, '0.lora.A':True, '0.lora.B':True, '1.1.linear.weight':False, '1.1.linear.bias':False, '1.1.lora.A':True, '1.1.lora.B':True}, (('0.linear.weight', None,), ('0.linear.bias', None,), ('0.lora.A', torch.tensor([[-0.2892272174358368], [0.11596192419528961], [0.1464923471212387]],dtype=torch.float32),), ('0.lora.B', torch.tensor([[0.059829868376255035, 0.025584250688552856, 0.07304096221923828, 0.49772271513938904]],dtype=torch.float32),), ('1.1.linear.weight', None,), ('1.1.linear.bias', None,), ('1.1.lora.A', torch.tensor([[-0.026296619325876236], [-0.03844499588012695], [-0.22954857349395752], [-0.17509889602661133]],dtype=torch.float32),), ('1.1.lora.B', torch.tensor([[-1.128781795501709, 1.958971381187439]],dtype=torch.float32),),),), (torch.tensor([[[-0.506294846534729, 0.7196618318557739], [-0.13468031585216522, 0.2675023078918457], [-0.3419768512248993, 0.554630696773529]], [[-0.3859787583351135, 0.5493986010551453], [0.13492770493030548, -0.029416389763355255], [0.03465476632118225, 0.06109250336885452]]],dtype=torch.float32), {'0.linear.weight':False, '0.linear.bias':False, '0.lora.A':True, '0.lora.B':True, '1.1.linear.weight':False, '1.1.linear.bias':False, '1.1.lora.A':True, '1.1.lora.B':True}, (('0.linear.weight', None,), ('0.linear.bias', None,), ('0.lora.A', torch.tensor([[-0.03219441697001457, -0.06748220324516296], [0.02932603284716606, 0.06213972717523575], [-0.060697730630636215, -0.12832745909690857]],dtype=torch.float32),), ('0.lora.B', torch.tensor([[0.0, 0.15513677895069122, 0.008122142404317856, 0.06438267230987549], [0.0, -0.0006719522061757743, 0.003330413717776537, -0.0029414482414722443]],dtype=torch.float32),), ('1.1.linear.weight', None,), ('1.1.linear.bias', None,), ('1.1.lora.A', torch.tensor([[0.0, 0.0], [0.02776217646896839, 0.038940973579883575], [0.0008769771666266024, 0.001343954703770578], [0.020260479301214218, 0.02856045588850975]],dtype=torch.float32),), ('1.1.lora.B', torch.tensor([[0.018285401165485382, -0.02803122252225876], [-0.18288135528564453, 0.27958357334136963]],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(2260+case)
        model=nn.Sequential(nn.Linear(3,4),nn.Sequential(nn.ReLU(),nn.Linear(4,2)))
        LORA_RANK=case+1
        LORA_ALPHA=2.
        actual=solve_26(model, LORA_RANK, LORA_ALPHA)
        observed=observe(actual,'freeze',case,locals())
        same(observed,expected[case])
    print('PASS · 26 · 두 입력 및 shape/dtype/동작 검증')
try:
    check_26()
except Exception as error:
    print('FAIL · 26 ·',type(error).__name__,str(error))


<details><summary>해설 · 쉬운 공식 · 참고 답안 (시도 후 펼치세요)</summary>

원본 파라미터 requires_grad=False → 새 LoRA 파라미터 생성. 새 A/B만 학습 가능해야 합니다. 순서를 바꾸면 LoRA까지 동결될 수 있습니다.

```python
def solve_26(model, LORA_RANK, LORA_ALPHA):
    for param in model.parameters():
        param.requires_grad = False
    replace_linear_with_lora(model, rank=LORA_RANK, alpha=LORA_ALPHA)
    return model
```

다운로드와 큰 실행 문맥은 제외했습니다. 참고 답안은 원본의 TODO 힌트에 근거합니다.
</details>